# Этап 1. Подготовка данных

Корпус проекта — 128 текстовых файлов из папки `texts/` и Excel-каталог с метаданными. В этом ноутбуке загружаем файлы, парсим метаданные, классифицируем тексты по языку и жанру, фильтруем нерелевантные, очищаем, формируем финальные датасеты и разбиваем на train/validation/test.

In [ ]:
# корневая папка проекта и пути к основным директориям
import os

BASE_DIR = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
DATASETS_DIR = os.path.join(BASE_DIR, "datasets")
TEMP_DIR = os.path.join(BASE_DIR, "temp", "visualization")

# создаём папки для результатов, если их ещё нет
os.makedirs(os.path.join(DATASETS_DIR, "processed"), exist_ok=True)
os.makedirs(os.path.join(DATASETS_DIR, "raw"), exist_ok=True)
os.makedirs(os.path.join(DATASETS_DIR, "cleaned_texts"), exist_ok=True)
os.makedirs(TEMP_DIR, exist_ok=True)

print(f"Корень проекта: {BASE_DIR}")
print(f"Датасеты: {DATASETS_DIR}")
print(f"Визуализации: {TEMP_DIR}")

## 1.1. Инвентаризация файлов

Сканируем папку `texts/`, собираем все `.txt`-файлы. Из имени каждого файла извлекаем номер, автора и название с помощью регулярных выражений. Для каждого файла определяем размер в килобайтах и кодировку через `chardet`.

### Импорты и настройки

In [ ]:
import re
from pathlib import Path

import chardet
import pandas as pd

# путь к папке с текстами
TEXTS_DIR = Path('texts')

# путь для сохранения итогового CSV
OUTPUT_CSV = Path('datasets/processed/inventory_raw.csv')

### Ручные исправления и вспомогательные функции

Часть файлов (в основном албаноязычные) имеют склеенные имена без разделителей. Для них задаём ручной маппинг автор/заголовок. Также нужна функция для разбиения CamelCase-строк на слова.

In [ ]:
# ручные исправления для файлов, которые невозможно разобрать автоматически
# ключ -- текст после номера (без расширения, без пробелов по краям)
# значение -- (автор, заголовок)
# нужны для албаноязычных файлов со склеенными именами и подчёркиваниями
MANUAL_OVERRIDES = {
    'LuanStarova': ('Luan Starova', ''),
    'NehasSopaj': ('Nehas Sopaj', ''),
    'AbdulkadirHyber': ('Abdulkadir Hyber', ''),
    'FahriKajaRaskazi': ('Fahri Kaja', 'Raskazi'),
    'Raskazi_Halil': ('Halil', 'Raskazi'),
    'IzborRaskazi_FejziBojku': ('Fejzi Bojku', 'Izbor Raskazi'),
    'IzborPOEZIJAi_FejziBojku': ('Fejzi Bojku', 'Izbor Poezija'),
    'IzborPoezija Abdulkadir Hayber': ('Abdulkadir Hayber', 'Izbor Poezija'),
    'AL PoezijaII Aliu': ('Aliu', 'Poezija II'),
    'AL- Poezija Aliu': ('Aliu', 'Poezija'),
    'Melahat Ali Detski Raskazi': ('Melahat Ali', 'Detski Raskazi'),
    'Sennur Suleyman Basar - Kritika': ('Sennur Suleyman Basar', 'Kritika'),
}


def split_camelcase(text):
    """разбиваем CamelCase-строку на слова, если она склеена без пробелов"""
    # пытаемся разбить по границе строчная->заглавная
    parts = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)
    # ещё разбиваем по границе несколько заглавных -> заглавная+строчная (для "ALiALiu")
    parts = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1 \2', parts)
    return parts.strip()

### Парсинг имени файла

Имена файлов в корпусе имеют разный формат: `06 а) nikola kirov majski - ilinden`, `15 Конески`, `38LuanStarova`. Функция `parse_filename` обрабатывает все варианты: извлекает номер (включая подпункты вроде `06а`), автора и заголовок.

In [ ]:
def parse_filename(filename):
    """
    парсим имя файла и достаём номер, автора и заголовок.
    возвращаем словарь с ключами: file_number, author, title
    """
    # убираем расширение .txt
    name = filename
    if name.endswith('.txt'):
        name = name[:-4]

    # убираем лишние пробелы в конце (бывают в некоторых файлах)
    name = name.rstrip()

    # пробуем извлечь номер с подпунктом, например "06 а)"
    # паттерн: цифры, потом опционально пробел + кириллическая буква + скобка
    match_sub = re.match(r'^(\d+)\s*([а-яёa-z])\)\s*(.*)$', name, re.IGNORECASE)
    if match_sub:
        # нашли подпункт: "06 а) nikola kirov majski - ilinden"
        file_number = match_sub.group(1) + match_sub.group(2)
        rest = match_sub.group(3).strip()
        author, title = _parse_author_title(rest)
        return {'file_number': file_number, 'author': author, 'title': title}

    # пробуем обычный номер: цифры в начале, потом пробел или сразу текст
    match_num = re.match(r'^(\d+)\s*(.*)', name)
    if match_num:
        file_number = match_num.group(1)
        rest = match_num.group(2).strip()
        author, title = _parse_author_title(rest)
        return {'file_number': file_number, 'author': author, 'title': title}

    # если номер не нашли (маловероятно), кладём всё в автора
    return {'file_number': '', 'author': name, 'title': ''}


def _parse_author_title(text):
    """
    из строки вида 'Автор - Заголовок' или 'Автор Заголовок' достаём автора и заголовок.
    возвращаем кортеж (author, title)
    """
    # если текст пустой
    if not text:
        return ('', '')

    # сначала проверяем, нет ли ручного исправления для этого текста
    if text in MANUAL_OVERRIDES:
        return MANUAL_OVERRIDES[text]

    # также проверяем текст без лишних пробелов на краях (на всякий случай)
    text_stripped = text.strip()
    if text_stripped in MANUAL_OVERRIDES:
        return MANUAL_OVERRIDES[text_stripped]

    # пробуем разделить по " - " (пробел-тире-пробел) -- самый чёткий разделитель
    if ' - ' in text:
        parts = text.split(' - ', 1)
        author_part = parts[0].strip()
        title_part = parts[1].strip()

        # проверяем, не жанровый ли префикс (DRAMA, AL, KRITIKA и т.п.)
        genre_prefixes = {'DRAMA', 'AL', 'KRITIKA'}
        if author_part.upper() in genre_prefixes:
            # "DRAMA - Skelzen Halimi" -> автор это правая часть
            return (title_part, '')

        return (author_part, title_part)

    # пробуем разделить по тире без пробелов ("KRITIKA-ALI ALIU", "AL-Poezija")
    if '-' in text:
        parts = text.split('-', 1)
        left = parts[0].strip()
        right = parts[1].strip()

        # если левая часть -- жанровый маркер, автор справа
        genre_prefixes = {'DRAMA', 'AL', 'KRITIKA'}
        if left.upper() in genre_prefixes:
            return (right, '')

        # обычное тире -- делим как автор/заголовок
        return (left, right)

    # нет тире -- пробуем другие стратегии

    # проверяем, не склеено ли CamelCase (типа "NehasSopaj", "AbdulkadirHyber")
    # условие: только латиница, без пробелов, и есть граница строчная-заглавная
    if re.match(r'^[A-Za-z]+$', text) and re.search(r'[a-z][A-Z]', text):
        expanded = split_camelcase(text)
        # возвращаем разбитое имя как автора, заголовок пустой
        return (expanded, '')

    # для файлов с коротким именем (1-2 слова) -- скорее всего это просто автор
    words = text.split()
    if len(words) <= 2:
        return (text, '')

    # для файлов с 3+ словами без тире -- эвристика: где кончается автор и начинается заголовок
    # список слов, которые точно не часть имени автора (предлоги, начало заголовка)
    non_name_words = {
        'По', 'На', 'Во', 'За', 'Од', 'До', 'Со', 'Кон',
        'Херменевтика', 'Кодот', 'Современа', 'Македонскиот',
        'Критика', 'Избор', 'Избрани',
        'Одбрани', 'Парадоксот', 'Огледало',
        'Раскази', 'Романи', 'Песни', 'Романот',
        'Poezija', 'PoezijaII', 'Raskazi', 'Kritika', 'Detski',
    }

    # если второе слово -- начало заголовка, автор = 1 слово
    if words[1] in non_name_words:
        return (words[0], ' '.join(words[1:]))

    # по умолчанию: автор = первые 2 слова, заголовок = остальное
    author = ' '.join(words[:2])
    title = ' '.join(words[2:]) if len(words) > 2 else ''

    return (author, title)

### Определение кодировки и размера файла

In [ ]:
def detect_encoding(filepath, sample_size=10000):
    """
    определяем кодировку файла через chardet.
    читаем первые sample_size байт для быстроты.
    возвращаем строку с названием кодировки.
    """
    # открываем файл в бинарном режиме
    with open(filepath, 'rb') as f:
        # считываем кусочек файла
        raw = f.read(sample_size)

    # chardet анализирует байты и выдаёт словарь с полем 'encoding'
    result = chardet.detect(raw)

    # берём название кодировки; если chardet не уверен, вернёт None
    encoding = result.get('encoding', 'unknown')
    return encoding if encoding else 'unknown'


def get_file_size_kb(filepath):
    """получаем размер файла в килобайтах, округляем до 2 знаков"""
    # stat() возвращает метаинформацию, st_size -- размер в байтах
    size_bytes = filepath.stat().st_size
    # переводим в KB
    return round(size_bytes / 1024, 2)

### Сканируем папку с текстами

In [ ]:
# собираем все файлы с расширением .txt
all_files = sorted(TEXTS_DIR.glob('*.txt'))
print(f'найдено .txt файлов: {len(all_files)}')
print()

# выводим список для проверки
for i, f in enumerate(all_files, 1):
    print(f'  {i:3d}. {f.name}')

### Парсим файлы: номер, автор, заголовок, размер, кодировка

In [ ]:
# список для накопления данных
rows = []

for filepath in all_files:
    # парсим имя файла
    parsed = parse_filename(filepath.name)

    # получаем размер файла в KB
    size_kb = get_file_size_kb(filepath)

    # определяем кодировку
    encoding = detect_encoding(filepath)

    # собираем строку для DataFrame
    row = {
        'file_number': parsed['file_number'],
        'file_name': filepath.name,
        'author': parsed['author'],
        'title': parsed['title'],
        'file_size_kb': size_kb,
        'encoding': encoding,
        'extension': filepath.suffix,
    }
    rows.append(row)

    # печатаем для лога
    print(
        f"  #{parsed['file_number']:>5s} | "
        f"автор: {parsed['author']:<35s} | "
        f"заголовок: {parsed['title']:<45s} | "
        f"{size_kb:>8.1f} KB | "
        f"encoding: {encoding}"
    )

### Собираем inventory DataFrame и смотрим статистику

In [ ]:
# создаём DataFrame из списка словарей
df_inv = pd.DataFrame(rows)

# выводим общую статистику
print(f'всего файлов в таблице: {len(df_inv)}')
print(f'уникальных авторов: {df_inv["author"].nunique()}')
print()

# статистика по размерам файлов
print('статистика по размерам файлов (KB):')
print(f'  минимум:  {df_inv["file_size_kb"].min():.1f} KB')
print(f'  максимум: {df_inv["file_size_kb"].max():.1f} KB')
print(f'  среднее:  {df_inv["file_size_kb"].mean():.1f} KB')
print(f'  медиана:  {df_inv["file_size_kb"].median():.1f} KB')
print(f'  суммарно: {df_inv["file_size_kb"].sum():.1f} KB ({df_inv["file_size_kb"].sum() / 1024:.1f} MB)')
print()

# статистика по кодировкам
print('распределение по кодировкам:')
encoding_counts = df_inv['encoding'].value_counts()
for enc, count in encoding_counts.items():
    print(f'  {enc}: {count} файлов')
print()

# показываем файлы без заголовка
no_title = df_inv[df_inv['title'] == '']
print(f'файлы без распознанного заголовка: {len(no_title)}')
for _, row in no_title.iterrows():
    print(f"  #{row['file_number']} {row['author']}")
print()

# показываем первые 10 строк таблицы
print('первые 10 строк таблицы:')
print(df_inv.head(10).to_string(index=False))

### Сохраняем inventory_raw.csv

In [ ]:
# сохраняем в CSV с кодировкой UTF-8 и BOM для корректного открытия в Excel
df_inv.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'файл сохранён: {OUTPUT_CSV}')
print(f'размер CSV: {OUTPUT_CSV.stat().st_size / 1024:.1f} KB')
print(f'строк: {len(df_inv)}, столбцов: {len(df_inv.columns)}')
print(f'столбцы: {list(df_inv.columns)}')

## 1.2. Анализ Excel-каталога

Читаем Excel-файл `texts/Naslovi_knigi_tabela .xls` с метаданными книг. Переименовываем столбцы из македонского в латиницу, анализируем заполненность каждого поля, нормализуем год оригинального издания.

### Читаем Excel-файл

In [ ]:
# путь к Excel-файлу (обрати внимание на пробел перед .xls -- так файл назван)
excel_path = 'texts/Naslovi_knigi_tabela .xls'

# путь для итогового CSV-каталога
csv_output_path = 'datasets/processed/catalog_from_excel.csv'

# читаем .xls через движок xlrd (старый формат Excel)
df_cat = pd.read_excel(excel_path, engine='xlrd')

print(f'прочитали {df_cat.shape[0]} строк и {df_cat.shape[1]} столбцов')
print(f'оригинальные имена столбцов: {list(df_cat.columns)}')
print()

# покажем первые 5 строк до переименования
print('первые 5 строк (до переименования):')
df_cat.head()

### Переименовываем столбцы в латиницу

In [ ]:
# маппинг: македонские названия -> латинские короткие имена
column_mapping = {
    'Код': 'number',                      # порядковый номер книги
    'Наслов': 'title_mk',                # название книги на македонском
    'Автор': 'author',                    # автор (фамилия имя)
    'Издавачка куќа': 'source',          # издательство (источник)
    'Година на издавање': 'year',         # год издания (в каталоге, 2008 или 2013)
    'Жанр': 'genre',                      # жанр: Проза, Поезија, Драма, Критика
    'Оригинално издание': 'original_year' # год оригинального издания (если известен)
}

# переименовываем столбцы
df_cat.rename(columns=column_mapping, inplace=True)

print(f'новые имена столбцов: {list(df_cat.columns)}')
print()

# покажем первые 5 строк после переименования
print('первые 5 строк (после переименования):')
df_cat.head()

### Анализ полноты данных

In [ ]:
# общее количество записей
total = len(df_cat)
print(f'всего записей: {total}')

# считаем заполненность каждого столбца
for col in ['original_year', 'genre', 'source', 'year', 'author', 'title_mk']:
    filled = df_cat[col].notna().sum()
    pct = filled / total * 100
    print(f'  заполнен {col}: {filled} из {total} ({pct:.1f}%)')
print()

# статистика по жанрам
print('распределение по жанрам:')
genre_counts = df_cat['genre'].value_counts()
for genre, count in genre_counts.items():
    print(f'  {genre}: {count} ({count / total * 100:.1f}%)')
print()

# статистика по источникам
print('распределение по источникам (издательствам):')
source_counts = df_cat['source'].value_counts()
for source, count in source_counts.items():
    print(f'  {source}: {count} ({count / total * 100:.1f}%)')
print()

# статистика по годам каталога
print('распределение по годам каталога:')
year_counts = df_cat['year'].value_counts().sort_index()
for yr, count in year_counts.items():
    print(f'  {yr}: {count} ({count / total * 100:.1f}%)')

### Нормализация года оригинального издания

В столбце `original_year` встречаются разные форматы: числа, строки вида `"1972 / 1981"`, просто `"/"`, строки вида `"1980 (Пиреј)"`. Извлекаем из каждого значения первый четырёхзначный год.

In [ ]:
# посмотрим, какие значения original_year содержат
print('уникальные значения original_year (до нормализации):')
for i, row in df_cat.iterrows():
    val = row['original_year']
    if pd.notna(val):
        print(f"  строка {i} (number={row['number']}): {repr(val)}")

In [ ]:
def extract_year(value):
    """извлекаем первый четырехзначный год из значения"""
    # если пусто -- возвращаем None
    if pd.isna(value):
        return None

    # если число -- сразу возвращаем как int
    if isinstance(value, (int, float)):
        return int(value)

    # преобразуем в строку
    s = str(value).strip()

    # если строка пустая или просто "/" -- нет данных
    if s in ('', '/'):
        return None

    # ищем первое четырехзначное число (год от 1900 до 2099)
    match = re.search(r'(1[89]\d{2}|20\d{2})', s)
    if match:
        return int(match.group(1))

    # если ничего не нашли -- вернем None
    return None


# применяем функцию к каждому значению original_year
df_cat['year_clean'] = df_cat['original_year'].apply(extract_year)

# переводим в nullable integer (Int64), чтобы NaN не превращались во float
df_cat['year_clean'] = df_cat['year_clean'].astype('Int64')

# смотрим результат нормализации
print('результат нормализации original_year -> year_clean:')
mask = df_cat['original_year'].notna()
print(df_cat.loc[mask, ['number', 'title_mk', 'original_year', 'year_clean']].to_string())
print()

# сколько строк получили корректный год после нормализации
filled_original_year = df_cat['original_year'].notna().sum()
clean_count = df_cat['year_clean'].notna().sum()
print(f'строк с original_year: {filled_original_year}')
print(f'строк с year_clean (после очистки): {clean_count}')
print(f'потеряно при очистке: {filled_original_year - clean_count} (это строки с "/" без года)')
print()

# диапазон годов
valid_years = df_cat['year_clean'].dropna()
if len(valid_years) > 0:
    print(f'минимальный год оригинала: {valid_years.min()}')
    print(f'максимальный год оригинала: {valid_years.max()}')
    print(f'медиана: {valid_years.median()}')

### Сохраняем каталог в CSV

In [ ]:
# финальный набор столбцов
final_columns = ['number', 'title_mk', 'author', 'source', 'year', 'genre', 'original_year', 'year_clean']
df_cat_final = df_cat[final_columns]

# выводим итоговую таблицу
print(f'итоговая таблица: {df_cat_final.shape[0]} строк, {df_cat_final.shape[1]} столбцов')
print(f'столбцы: {list(df_cat_final.columns)}')
print()

# сохраняем CSV с UTF-8 кодировкой и BOM (чтобы Excel тоже открывал нормально)
df_cat_final.to_csv(csv_output_path, index=False, encoding='utf-8-sig')
print(f'CSV сохранен: {csv_output_path}')

# проверяем, что файл можно обратно прочитать
df_check = pd.read_csv(csv_output_path, encoding='utf-8-sig')
print(f'проверка: прочитано обратно {df_check.shape[0]} строк, {df_check.shape[1]} столбцов')
print()

# последние 5 строк итоговой таблицы для проверки
print('последние 5 строк итоговой таблицы:')
df_cat_final.tail()

## 1.3. Объединение инвентаризации и каталога

Объединяем таблицу инвентаризации (файлы с диска) с каталогом (из Excel) через merge по номеру файла. Ищем расхождения: какие файлы есть на диске, но отсутствуют в каталоге, и наоборот.

### Загружаем обе таблицы

In [ ]:
# загружаем таблицу инвентаризации (сканирование файлов, шаг 1)
inv = pd.read_csv(
    'datasets/processed/inventory_raw.csv',
    encoding='utf-8-sig',
    dtype={'file_number': str}
)
print(f'инвентаризация: {len(inv)} строк')
print(f'столбцы: {list(inv.columns)}')

# загружаем каталог из Excel (шаг 2)
cat = pd.read_csv(
    'datasets/processed/catalog_from_excel.csv',
    encoding='utf-8-sig',
    dtype={'number': str}
)
print(f'каталог: {len(cat)} строк')
print(f'столбцы: {list(cat.columns)}')

### Создаём числовые ключи для merge

В инвентаризации номера файлов бывают с буквенными суффиксами: `06а`, `06б`, `06в`, `06г`. В каталоге им соответствует одна запись с номером `6`. Для корректного merge нужен числовой ключ без суффиксов. Также в каталоге есть запись `94/95` -- раскрываем её в две строки.

In [ ]:
def extract_numeric_key(file_num):
    """убираем буквенные суффиксы (а, б, в, г) из номера файла"""
    # "06а" -> "6", "06б" -> "6", "10" -> "10"
    match = re.match(r'^0*(\d+)', str(file_num))
    if match:
        return match.group(1)
    return str(file_num)


def normalize_catalog_key(num):
    """убираем ведущие нули из номера каталога"""
    match = re.match(r'^0*(\d+)', str(num))
    if match:
        return match.group(1)
    return str(num)


# добавляем merge_key в таблицу инвентаризации
inv['merge_key'] = inv['file_number'].apply(extract_numeric_key)

# показываем файлы с номерами 06* для проверки
print('примеры merge_key из инвентаризации:')
mask_06 = inv['file_number'].str.startswith('06')
print(inv.loc[mask_06, ['file_number', 'merge_key', 'author']].to_string(index=False))

In [ ]:
# обрабатываем строку "94/95" в каталоге: разбиваем на две строки
cat_expanded = []
for _, row in cat.iterrows():
    num = str(row['number'])
    if '/' in num:
        # разбиваем "94/95" на отдельные строки
        parts = num.split('/')
        for part in parts:
            new_row = row.copy()
            new_row['merge_key'] = normalize_catalog_key(part.strip())
            new_row['number_original'] = num
            cat_expanded.append(new_row)
    else:
        row_copy = row.copy()
        row_copy['merge_key'] = normalize_catalog_key(num)
        row_copy['number_original'] = num
        cat_expanded.append(row_copy)

cat_exp = pd.DataFrame(cat_expanded)
print(f'каталог после раскрытия "94/95": {len(cat_exp)} строк')

### Выполняем merge и анализируем расхождения

In [ ]:
# выполняем merge (left = инвентаризация, right = каталог)
merged = inv.merge(
    cat_exp,
    on='merge_key',
    how='outer',
    suffixes=('_file', '_catalog'),
    indicator=True
)

print(f'результат merge: {len(merged)} строк')
print()
print('распределение по типу merge:')
print(merged['_merge'].value_counts().to_string())
print()

# файлы на диске, которых нет в каталоге
only_disk = merged[merged['_merge'] == 'left_only']
print(f'файлы на диске, которых нет в каталоге ({len(only_disk)}):')
if len(only_disk) > 0:
    print(only_disk[['file_number', 'file_name', 'author_file']].to_string(index=False))
print()

# записи каталога, для которых нет файла на диске
only_catalog = merged[merged['_merge'] == 'right_only']
print(f'записи каталога без файла на диске ({len(only_catalog)}):')
if len(only_catalog) > 0:
    print(only_catalog[['merge_key', 'title_mk', 'author_catalog', 'genre']].to_string(index=False))
print()

# записи, которые совпали
both = merged[merged['_merge'] == 'both']
print(f'совпавшие записи: {len(both)}')

### Сравнение авторов из файлов и каталога

In [ ]:
# сравниваем авторов между именем файла и каталогом
print('сравнение авторов (первые 30 записей):')
comparison = both[['merge_key', 'author_file', 'author_catalog', 'title', 'title_mk']].head(30)
print(comparison.to_string(index=False))

### Формируем и сохраняем финальную объединённую таблицу

In [ ]:
# формируем финальную таблицу: берём все файлы с диска, добавляем данные каталога
# для файлов без записи в каталоге -- столбцы каталога будут пустыми
final = merged.copy()

# переименовываем столбцы для удобства
final = final.rename(columns={
    'author_file': 'author_from_filename',
    'author_catalog': 'author_from_catalog',
    'title': 'title_from_filename',
    'title_mk': 'title_from_catalog',
})

# выбираем столбцы для финальной таблицы
final_columns = [
    'file_number',            # номер файла из имени
    'merge_key',              # числовой ключ для связи
    'file_name',              # полное имя файла
    'author_from_filename',   # автор из имени файла
    'author_from_catalog',    # автор из каталога
    'title_from_filename',    # название из имени файла
    'title_from_catalog',     # название из каталога
    'genre',                  # жанр из каталога
    'source',                 # источник из каталога
    'year',                   # год издания (каталог)
    'year_clean',             # нормализованный год
    'original_year',          # оригинальный год
    'file_size_kb',           # размер файла в KB
    'encoding',               # кодировка файла
    '_merge',                 # тип merge (для отладки)
]

# проверяем, что все столбцы есть
available = [c for c in final_columns if c in final.columns]
missing = [c for c in final_columns if c not in final.columns]
if missing:
    print(f'отсутствуют столбцы: {missing}')

final = final[available]

# сортируем по merge_key (числовой порядок)
final['sort_key'] = final['merge_key'].apply(lambda x: int(x) if x.isdigit() else 999)
final = final.sort_values(['sort_key', 'file_number']).drop(columns='sort_key')

# сохраняем объединённую таблицу
output_path = 'datasets/processed/inventory_merged.csv'
final.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'сохранено в {output_path}')
print(f'итого строк: {len(final)}')
print(f'столбцов: {len(final.columns)}')

### Итоговая статистика

In [ ]:
# сколько файлов с диска нашли запись в каталоге
n_matched = len(final[final['_merge'] == 'both'])
n_disk_only = len(final[final['_merge'] == 'left_only'])
n_catalog_only = len(final[final['_merge'] == 'right_only'])
print(f'совпадений (файл + каталог): {n_matched}')
print(f'только на диске (нет в каталоге): {n_disk_only}')
print(f'только в каталоге (нет файла): {n_catalog_only}')
print()

# распределение по жанрам (для совпавших)
print('распределение по жанрам (совпавшие записи):')
genre_counts = final[final['_merge'] == 'both']['genre'].value_counts()
print(genre_counts.to_string())
print()

# размер корпуса
total_size_mb = final['file_size_kb'].sum() / 1024
print(f'общий размер файлов: {total_size_mb:.1f} MB')

## Промежуточное резюме

После инвентаризации и объединения с каталогом получили:

- **128 файлов** на диске нашли соответствие в каталоге
- **10 записей** каталога не имеют файла на диске (нумерация в каталоге идёт до 138, но часть файлов отсутствует)
- Итого **138 строк** в объединённой таблице `inventory_merged.csv`
- Каждая строка содержит: номер файла, автора (из имени файла и из каталога), заголовок, жанр, год, размер, кодировку

Дальше будем классифицировать тексты по языку и фильтровать нерелевантные.

## 1.4. Определение языка текстов

Корпус содержит 128 файлов, но не все из них на македонском языке: среди них могут оказаться албанские и турецкие тексты.
Для каждого файла определяем язык по характерным символам кириллицы и латиницы.
Македонская кириллица содержит уникальные буквы (ѓ, ќ, љ, њ, џ), а албанский и турецкий используют латиницу с диакритиками (ë, ç, ş, ğ).
Дополнительно проверяем результат через библиотеку `langdetect`.

In [ ]:
# импортируем langdetect для автоматического определения языка
from langdetect import detect, DetectorFactory

# фиксируем seed для воспроизводимости результатов langdetect
DetectorFactory.seed = 0

# путь к папке с текстами
texts_dir = os.path.join(project_root, 'texts')

# путь к выходному CSV-файлу
lang_output_path = os.path.join(project_root, 'temp', 'language_detection.csv')

# загружаем таблицу inventory_merged.csv
inventory_path = os.path.join(project_root, 'datasets', 'inventory_merged.csv')
df_inv = pd.read_csv(inventory_path)

# оставляем только строки, где _merge == 'both' (файлы, найденные и в каталоге, и на диске)
df_both = df_inv[df_inv['_merge'] == 'both'].copy()

# проверяем, сколько строк получилось
print(f"Строк с _merge=='both': {len(df_both)}")

In [ ]:
# специфические македонские кириллические буквы, которых нет в сербском/болгарском
mk_specific_chars = set('ѓќљњџЃЌЉЊЏ')

# символы, встречающиеся только в албанском (ë уникальна для албанского)
sq_unique_chars = set('ëË')

# символы, общие для албанского и турецкого (ç/Ç)
sq_tr_shared_chars = set('çÇ')

# специфические турецкие символы (которых нет в албанском)
tr_specific_chars = set('şŞğĞıİöÖüÜ')

# все "не-mk" латинские диакритики (албанские + турецкие) для столбца has_sq_chars
sq_specific_chars = set('ëçËÇ')


def is_cyrillic(ch):
    """проверяем, кириллический ли символ (Unicode-диапазон 0x0400-0x04FF)"""
    code = ord(ch)
    return 0x0400 <= code <= 0x04FF


def is_latin(ch):
    """проверяем, латинский ли символ (базовая и расширенная латиница)"""
    code = ord(ch)
    # базовая латиница: A-Z, a-z
    basic = (0x0041 <= code <= 0x005A) or (0x0061 <= code <= 0x007A)
    # расширенная латиница (с диакритиками)
    extended = (0x00C0 <= code <= 0x024F)
    return basic or extended

In [ ]:
def detect_language(file_name):
    """анализируем один файл и определяем его язык по символам + langdetect"""

    # полный путь к файлу
    file_path = os.path.join(texts_dir, file_name)

    # проверяем, что файл существует
    if not os.path.exists(file_path):
        return {
            'language': 'error',
            'cyrillic_ratio': None,
            'has_mk_chars': None,
            'has_sq_chars': None,
            'detection_method': 'file_not_found'
        }

    # читаем первые 2000 символов файла в encoding utf-8-sig
    try:
        with open(file_path, 'r', encoding='utf-8-sig') as f:
            text = f.read(2000)
    except Exception as e:
        return {
            'language': 'error',
            'cyrillic_ratio': None,
            'has_mk_chars': None,
            'has_sq_chars': None,
            'detection_method': f'read_error: {e}'
        }

    # считаем количество кириллических и латинских букв
    cyrillic_count = sum(1 for ch in text if is_cyrillic(ch))
    latin_count = sum(1 for ch in text if is_latin(ch))

    # общее количество буквенных символов (кириллица + латиница)
    total_letters = cyrillic_count + latin_count

    # доля кириллицы среди всех букв (0.0 - 1.0)
    cyrillic_ratio = round(cyrillic_count / total_letters, 4) if total_letters > 0 else 0.0

    # проверяем наличие специфических македонских букв
    has_mk_chars = any(ch in mk_specific_chars for ch in text)

    # проверяем наличие специфических албанских символов (ë, ç и т.д.)
    has_sq_chars = any(ch in sq_specific_chars for ch in text)

    # проверяем наличие уникальных албанских символов (ë - только в албанском)
    has_sq_unique = any(ch in sq_unique_chars for ch in text)

    # проверяем наличие специфических турецких символов (ş, ğ, ı, ö, ü и т.д.)
    has_tr_chars = any(ch in tr_specific_chars for ch in text)

    # пробуем langdetect для дополнительной проверки
    langdetect_result = None
    try:
        langdetect_result = detect(text)
    except Exception:
        # langdetect может упасть на слишком коротком тексте
        langdetect_result = 'unknown'

    # логика определения языка
    # если почти вся латиница (кириллицы < 10%) - скорее всего не македонский
    if cyrillic_ratio < 0.10:
        # сначала проверяем, не турецкий ли это текст
        if has_tr_chars or langdetect_result == 'tr':
            # есть турецкие символы (ş, ğ, ı, ö, ü) или langdetect говорит "tr"
            language = 'tr'
            method = f'latin_dominant + tr_chars={has_tr_chars} + langdetect={langdetect_result}'
        # если есть уникальные албанские символы (ë) или langdetect говорит "sq"
        elif has_sq_unique or langdetect_result == 'sq':
            language = 'sq'
            method = f'latin_dominant + sq_unique={has_sq_unique} + langdetect={langdetect_result}'
        # если есть общие sq/tr символы (ç), но langdetect не определил ни sq ни tr
        elif has_sq_chars:
            # ç без ë и без турецких символов - неоднозначно, смотрим langdetect
            language = 'other'
            method = f'latin_dominant + ambiguous_diacritics + langdetect={langdetect_result}'
        else:
            # латиница без диакритик - помечаем как other
            language = 'other'
            method = f'latin_dominant + no_diacritics + langdetect={langdetect_result}'

    # если преобладает кириллица (> 90%)
    elif cyrillic_ratio > 0.90:
        # если есть специфические македонские буквы - точно mk
        if has_mk_chars:
            language = 'mk'
            method = 'cyrillic_dominant + mk_specific_chars'
        else:
            # кириллица без специфических mk-букв - может быть болгарский/сербский
            # но в контексте корпуса скорее всего все равно mk
            if langdetect_result in ('mk', 'bg', 'sr'):
                language = 'mk'
                method = f'cyrillic_dominant + langdetect={langdetect_result}'
            else:
                language = 'mk'
                method = f'cyrillic_dominant + langdetect={langdetect_result} (assumed mk)'

    # если смешанный текст (10-90% кириллицы)
    else:
        # если есть македонские буквы - mk с латинскими вкраплениями
        if has_mk_chars and cyrillic_ratio > 0.5:
            language = 'mk'
            method = f'mixed + mk_chars + cyrillic={cyrillic_ratio}'
        # если есть албанские символы - скорее sq
        elif has_sq_chars and cyrillic_ratio < 0.5:
            language = 'sq'
            method = f'mixed + sq_chars + cyrillic={cyrillic_ratio}'
        # в остальных случаях ориентируемся на долю кириллицы + langdetect
        elif cyrillic_ratio > 0.5:
            language = 'mk'
            method = f'mixed + cyrillic_majority={cyrillic_ratio} + langdetect={langdetect_result}'
        else:
            language = 'other'
            method = f'mixed + latin_majority={cyrillic_ratio} + langdetect={langdetect_result}'

    return {
        'language': language,
        'cyrillic_ratio': cyrillic_ratio,
        'has_mk_chars': has_mk_chars,
        'has_sq_chars': has_sq_chars,
        'detection_method': method,
        'langdetect_result': langdetect_result
    }

In [ ]:
# список для сбора результатов
lang_results = []

# проходим по каждому файлу из отфильтрованной таблицы
for _, row in df_both.iterrows():
    # имя файла из таблицы
    file_name = row['file_name']
    # номер файла из таблицы
    file_number = row['file_number']

    # запускаем определение языка
    detection = detect_language(file_name)

    # собираем строку результата
    lang_results.append({
        'file_number': file_number,
        'file_name': file_name,
        'language': detection['language'],
        'cyrillic_ratio': detection['cyrillic_ratio'],
        'has_mk_chars': detection['has_mk_chars'],
        'has_sq_chars': detection['has_sq_chars'],
        'detection_method': detection['detection_method'],
        'langdetect_result': detection.get('langdetect_result', '')
    })

# собираем результаты в DataFrame
lang_results_df = pd.DataFrame(lang_results)

# сохраняем в CSV
lang_results_df.to_csv(lang_output_path, index=False, encoding='utf-8-sig')
print(f"Результаты сохранены в: {lang_output_path}")

In [ ]:
# выводим сводку по языкам
print("Сводка по языкам:")
lang_counts = lang_results_df['language'].value_counts()
for lang, count in lang_counts.items():
    print(f"  {lang}: {count} файлов")
print(f"  ИТОГО: {len(lang_results_df)} файлов")

# выводим список всех НЕ-mk файлов
non_mk = lang_results_df[lang_results_df['language'] != 'mk']
print(f"\nФайлы не на македонском ({len(non_mk)} шт.):")
for _, row in non_mk.iterrows():
    print(f"  [{row['language']}] {row['file_number']} | {row['file_name']}")
    print(f"         кириллица: {row['cyrillic_ratio']}, mk-буквы: {row['has_mk_chars']}, sq-буквы: {row['has_sq_chars']}")
    print(f"         метод: {row['detection_method']}")
    print(f"         langdetect: {row['langdetect_result']}")

## 1.5. Классификация по жанру

Используем жанр из каталога как отправную точку, но дополняем его эвристиками по тексту.
Для драмы ищем характерные маркеры: ремарки в скобках, реплики в формате `ИМЯ ПЕРСОНАЖА:`, заголовки `ДЕЈСТВИЕ`, `СЦЕНА`.
Для поэзии ориентируемся на короткие строки (медиана длины строки < 45 символов).
Также отдельно помечаем детскую литературу и критику/эссеистику, исправляя ошибки каталога.

In [ ]:
import sys

# принудительно выставляем utf-8 для вывода в консоль (Windows)
sys.stdout.reconfigure(encoding='utf-8')

# путь к таблице с метаданными
genre_inventory_path = os.path.join(project_root, 'datasets', 'inventory_merged.csv')

# путь к папке с текстами
genre_texts_dir = os.path.join(project_root, 'texts')

# путь для сохранения результата
genre_output_path = os.path.join(project_root, 'temp', 'genre_classification.csv')

# загружаем таблицу с метаданными
genre_df = pd.read_csv(genre_inventory_path)

# оставляем только строки, где файл реально найден (и в каталоге, и на диске)
genre_df = genre_df[genre_df['_merge'] == 'both'].copy()

# сбрасываем индекс после фильтрации
genre_df = genre_df.reset_index(drop=True)

# сколько файлов получилось после фильтрации
print(f'файлов с _merge=both: {len(genre_df)}')

# уникальные жанры из каталога, чтобы понимать что есть
print(f'\nжанры из каталога: {genre_df["genre"].unique()}')
print(f'распределение жанров из каталога:')
print(genre_df['genre'].value_counts().to_string())

In [ ]:
def detect_genre_from_text(file_path, file_number, genre_catalog, title_catalog, title_filename):
    """
    читаем текст файла и применяем эвристики для определения жанра
    возвращаем кортеж: (жанр_по_эвристике, заметки_об_определении)
    """

    # пробуем прочитать файл
    try:
        # кодировка utf-8-sig (с BOM-маркером)
        with open(file_path, 'r', encoding='utf-8-sig') as f:
            text = f.read()
    except Exception as e:
        # если не удалось прочитать, возвращаем жанр из каталога
        return genre_catalog, f'ошибка чтения файла: {e}'

    # разбиваем текст на строки
    lines = text.split('\n')

    # убираем полностью пустые строки для анализа длин
    non_empty_lines = [line for line in lines if line.strip()]

    # если строк нет, возвращаем жанр из каталога
    if len(non_empty_lines) == 0:
        return genre_catalog, 'файл пустой'

    # считаем среднюю длину непустых строк (нужно для определения поэзии)
    avg_line_length = sum(len(line.strip()) for line in non_empty_lines) / len(non_empty_lines)

    # считаем медианную длину строк (устойчивее к выбросам)
    line_lengths = sorted(len(line.strip()) for line in non_empty_lines)
    median_line_length = line_lengths[len(line_lengths) // 2]

    # берем первые 5000 символов для поиска ключевых слов (шапка текста)
    header_text = text[:5000]

    # заметки для отчета
    notes = []

    # маркеры драмы: ищем как отдельные фразы, а не подстроки
    drama_header_patterns = [
        # "Драма во N дејствија/дела" - самый надежный маркер
        r'[Дд]рама\s+во\s+(два|три|четири|пет|шест)',
        # "ДЕЈСТВИЕ ПРВО/ВТОРО/..." или "ДЕЈСТВИЕ I/II/..."
        r'ДЕЈСТВИ[ЕЈ]\s+(ПРВО|ВТОРО|ТРЕТО|ЧЕТВРТО|ПЕТТО|I|II|III|IV|V|\d)',
        # "ДЕЈСТВО ПРВО/ВТОРО/..." или "ДЕЈСТВО I/II/..."
        r'ДЕЈСТВО\s+(ПРВО|ВТОРО|ТРЕТО|ЧЕТВРТО|ПЕТТО|I|II|III|IV|V|\d)',
        # "ЧИН ПРВО/ВТОРО/..." - акт пьесы
        r'ЧИН\s+(ПРВО|ПРВО|ПРИВ|ВТОР|ТРЕТ|ЧЕТВРТ|ПЕТТ|I|II|III|IV|V|\d)',
        # "СЦЕНА" как отдельное слово с номером
        r'СЦЕНА\s+(ПРВА|ВТОРА|ТРЕТА|ЧЕТВРТА|ПЕТТА|I|II|III|IV|V|\d)',
        # "ЛИЦА:" - заголовок списка действующих лиц (на отдельной строке)
        r'^\s*ЛИЦА\s*[:\n]',
        # "Комедија" / "Трагедија" как часть описания жанра
        r'[Кк]омедија\s+(во|на)',
        r'[Тт]рагедија\s+(во|на)',
        # судска драма, музичка драма и т.п.
        r'[Сс]удска\s+драма',
        r'[Мм]узичка\s+драма',
    ]

    # маркеры драмы по всему тексту (для подсчета количества)
    drama_body_patterns = [
        # "ДЕЈСТВИЕ" с номером - встречается в теле драмы
        r'ДЕЈСТВИ[ЕЈ]\s+(ПРВО|ВТОРО|ТРЕТО|ЧЕТВРТО|ПЕТТО|I|II|III|IV|V|\d)',
        # "ДЕЈСТВО" с номером
        r'ДЕЈСТВО\s+(ПРВО|ВТОРО|ТРЕТО|ЧЕТВРТО|ПЕТТО|I|II|III|IV|V|\d)',
        # "СЦЕНА" с номером
        r'СЦЕНА\s+(ПРВА|ВТОРА|ТРЕТА|ЧЕТВРТА|ПЕТТА|I|II|III|IV|V|\d)',
        # "ЧИН" с номером
        r'ЧИН\s+(ПРВО|ПРИВ|ВТОР|ТРЕТ|ЧЕТВРТ|ПЕТТ|I|II|III|IV|V|\d)',
    ]

    # считаем маркеры драмы в шапке текста
    drama_markers_in_header = []
    for pattern in drama_header_patterns:
        if re.search(pattern, header_text, re.MULTILINE | re.IGNORECASE):
            drama_markers_in_header.append(pattern[:30])

    # считаем маркеры драмы по всему тексту
    drama_body_count = 0
    for pattern in drama_body_patterns:
        drama_body_count += len(re.findall(pattern, text, re.MULTILINE))

    # ремарки: текст в скобках длиной >= 15 символов (настоящие ремарки)
    stage_directions = re.findall(r'\([^)]{15,}\)', text)
    stage_direction_count = len(stage_directions)

    # считаем строки формата "ИМЯ ПЕРСОНАЖА: реплика" (характерно для драмы)
    dialogue_pattern = re.compile(
        r'^\s*[А-ЯЃЅЌЉЊЏЈЖШЧЦ][А-ЯЃЅЌЉЊЏЈЖШЧЦ\s]{1,25}[:\.]',
        re.MULTILINE
    )
    dialogue_lines = dialogue_pattern.findall(text)
    dialogue_count = len(dialogue_lines)

    # файлы 94-97: детская литература (в названиях "за деца")
    file_num_str = str(file_number)
    is_children = False
    # проверяем по номеру файла
    if file_num_str in ['94', '95', '96', '97']:
        is_children = True
        notes.append(f'файл {file_num_str}: детская литература по номеру')

    # проверяем по названию (за деца / Detski / для детей / cocuklara)
    title_combined = str(title_catalog) + ' ' + str(title_filename)
    if re.search(r'за деца|за дец|detsk|[Cc][Cc]ocuk|[Gg][Gg]ocuk', title_combined, re.IGNORECASE):
        is_children = True
        notes.append('в названии есть маркер детской литературы')

    # файлы 128-129: тоже детские (по названиям из каталога: Cocuklara...)
    if file_num_str in ['128', '129']:
        tc_lower = str(title_catalog).lower()
        if 'cocuk' in tc_lower or 'çocuk' in tc_lower:
            is_children = True
            notes.append(f'файл {file_num_str}: детская литература по названию из каталога')

    # файлы 102-106: критика/эссеистика
    is_criticism = False
    if file_num_str in ['102', '103', '104', '105', '106']:
        is_criticism = True
        notes.append(f'файл {file_num_str}: критика/эссеистика по номеру и содержанию')

    # проверяем по названию (критика, херменевтика, есеј, есеи)
    criticism_markers = ['критика', 'херменевтика', 'есеј', 'есеи', 'критики и огледи',
                         'кодот на', 'eleştiriler', 'kritika']
    for marker in criticism_markers:
        if marker.lower() in title_combined.lower():
            is_criticism = True
            notes.append(f'в названии найден маркер критики: "{marker}"')

    # файлы 125 и 130: тоже критика по каталогу
    if genre_catalog == 'Критика':
        # если каталог говорит "Критика", доверяем, кроме файлов 06а-06г
        if file_num_str not in ['06а', '06б', '06в', '06г']:
            is_criticism = True
            notes.append('каталог: Критика')

    # определяем, помечен ли текст как драма
    is_drama = False

    # маркеры в шапке текста: самый надежный признак
    if len(drama_markers_in_header) >= 1:
        is_drama = True
        notes.append(f'маркеры драмы в шапке ({len(drama_markers_in_header)}): {drama_markers_in_header}')

    # много маркеров в теле текста = тоже драма
    if drama_body_count >= 3:
        is_drama = True
        notes.append(f'маркеры драмы в теле текста: {drama_body_count}')

    # много ремарок + много диалогов = тоже драма
    if stage_direction_count > 30 and dialogue_count > 40:
        is_drama = True
        notes.append(f'ремарок: {stage_direction_count}, диалогов в формате ИМЯ: {dialogue_count}')

    # жанр "Драма" из каталога тоже учитываем
    if genre_catalog == 'Драма':
        is_drama = True
        notes.append('каталог: Драма')

    # для файлов 06а-06г: каталог говорит "Критика" (потому что запись #6 это антология)
    # но на самом деле это пьесы, проверяем по маркерам в тексте и по title_catalog
    if file_num_str in ['06а', '06б', '06в', '06г']:
        notes.append(f'файл {file_num_str}: часть антологии "Македонска битова драма", каталожный жанр = Критика')
        # если нашли маркеры драмы в тексте или в title_catalog есть "драма"
        if is_drama or 'драма' in str(title_catalog).lower():
            is_drama = True
            notes.append('исправлено на Драма (пьеса из антологии)')

    # определяем поэзию: короткие строки
    is_poetry = False
    if median_line_length < 45 and avg_line_length < 50:
        is_poetry = True
        notes.append(f'средняя длина строки: {avg_line_length:.1f}, медиана: {median_line_length}')

    # если каталог говорит Поезија, тоже учитываем
    if genre_catalog == 'Поезија':
        is_poetry = True
        notes.append('каталог: Поезија')

    # принимаем решение в порядке приоритета
    if is_children:
        detected_genre = 'Детска книжевност'
    elif is_drama and not is_criticism:
        detected_genre = 'Драма'
    elif is_criticism:
        detected_genre = 'Критика / есеистика'
    elif is_poetry and not is_drama:
        detected_genre = 'Поезија'
    else:
        detected_genre = 'Проза'

    # собираем заметки в одну строку
    notes_str = '; '.join(notes) if notes else 'жанр из каталога совпал или нет особых маркеров'

    return detected_genre, notes_str

In [ ]:
# собираем результаты классификации для каждого файла
genre_results = []

for idx, row in genre_df.iterrows():
    # номер файла
    file_number = row['file_number']

    # имя файла
    file_name = row['file_name']

    # жанр из каталога
    genre_catalog = row['genre']

    # названия из каталога и из имени файла
    title_catalog = row.get('title_from_catalog', '')
    title_filename = row.get('title_from_filename', '')

    # полный путь к файлу
    file_path = os.path.join(genre_texts_dir, file_name)

    # проверяем что файл реально на диске
    if not os.path.exists(file_path):
        # файл не найден, пропускаем
        genre_results.append({
            'file_number': file_number,
            'file_name': file_name,
            'genre_catalog': genre_catalog,
            'genre_detected': genre_catalog,
            'genre_final': genre_catalog,
            'detection_notes': 'файл не найден на диске'
        })
        continue

    # определяем жанр по тексту
    genre_detected, detection_notes = detect_genre_from_text(
        file_path, file_number, genre_catalog, title_catalog, title_filename
    )

    # итоговый жанр: берем жанр по эвристике (он уже учитывает каталог)
    genre_final = genre_detected

    # собираем строку результата
    genre_results.append({
        'file_number': file_number,
        'file_name': file_name,
        'genre_catalog': genre_catalog,
        'genre_detected': genre_detected,
        'genre_final': genre_final,
        'detection_notes': detection_notes
    })

# создаем DataFrame с результатами
genre_result_df = pd.DataFrame(genre_results)

# сохраняем в CSV
genre_result_df.to_csv(genre_output_path, index=False, encoding='utf-8-sig')
print(f'\nрезультат сохранен в {genre_output_path}')

In [ ]:
# выводим итоговое распределение по жанрам
print('распределение по genre_final:')
print(genre_result_df['genre_final'].value_counts().to_string())

# для сравнения -- распределение из каталога
print('\nраспределение по genre_catalog (для сравнения):')
print(genre_result_df['genre_catalog'].value_counts().to_string())

# выводим файлы, где жанр был исправлен (отличается от каталога)
changed = genre_result_df[genre_result_df['genre_catalog'] != genre_result_df['genre_final']]
print(f'\nфайлы, где жанр исправлен ({len(changed)} шт.):')
for _, row in changed.iterrows():
    fn = row['file_name']
    # обрезаем имя файла для красивого вывода
    fn_short = fn[:60] if len(fn) <= 60 else fn[:57] + '...'
    print(f'  {row["file_number"]:>5}  {fn_short:<60}  '
          f'{row["genre_catalog"]:>20} -> {row["genre_final"]}')
    # печатаем заметки для понимания причины (обрезаем до 150 символов)
    notes_short = row["detection_notes"][:150]
    print(f'         причина: {notes_short}')

## 1.6. Объединение результатов и фильтрация корпуса

Объединяем языковую и жанровую классификацию в одну таблицу.
Каждому тексту присваиваем категорию: оригинальный македонский, албанский, турецкий или критика/эссеистика.
Для обучения нейросети нужны только оригинальные македонские художественные тексты, все остальное отфильтровываем.

In [ ]:
# загружаем основную таблицу с метаданными
merged_df = pd.read_csv(os.path.join(project_root, 'datasets', 'inventory_merged.csv'))

# оставляем только строки, где файл реально есть на диске
merged_df = merged_df[merged_df['_merge'] == 'both'].copy()

# сколько файлов будем обрабатывать
print(f'файлов с _merge == both: {len(merged_df)}')

# загружаем результаты определения языка
lang_df = pd.read_csv(os.path.join(project_root, 'temp', 'language_detection.csv'))

# загружаем результаты классификации жанров
genre_cls_df = pd.read_csv(os.path.join(project_root, 'temp', 'genre_classification.csv'))

# приводим file_number к строке в обеих таблицах, чтобы merge не сломался
merged_df['file_number'] = merged_df['file_number'].astype(str)
lang_df['file_number'] = lang_df['file_number'].astype(str)
genre_cls_df['file_number'] = genre_cls_df['file_number'].astype(str)

In [ ]:
# присоединяем столбцы языка к основной таблице
result_df = merged_df.merge(
    # берем только нужные столбцы из таблицы языков
    lang_df[['file_number', 'language', 'cyrillic_ratio', 'has_mk_chars',
             'has_sq_chars', 'detection_method', 'langdetect_result']],
    on='file_number',
    how='left'
)

# присоединяем столбцы жанра к результату
result_df = result_df.merge(
    # берем только нужные столбцы из таблицы жанров
    genre_cls_df[['file_number', 'genre_detected', 'genre_final', 'detection_notes']],
    on='file_number',
    how='left'
)

# проверяем, что ничего не потерялось при merge
print(f'строк после объединения: {len(result_df)}')

In [ ]:
# определяем, художественная ли это литература (fiction)
# критика и эссеистика -- не художественная
result_df['is_fiction'] = ~result_df['genre_final'].str.contains(
    'Критика', case=False, na=False
)

# текст считаем оригинальным македонским, если язык mk и это fiction
result_df['is_original_mk'] = (
    (result_df['language'] == 'mk') & result_df['is_fiction']
)


def classify_text(row):
    """определяем категорию текста по языку и жанру"""
    # если язык албанский -- помечаем как albanian
    if row['language'] == 'sq':
        return 'albanian'
    # если язык турецкий -- помечаем как turkish
    if row['language'] == 'tr':
        return 'turkish'
    # если язык не македонский -- помечаем как other_language
    if row['language'] != 'mk':
        return 'other_language'
    # если это критика/эссеистика -- помечаем как non_fiction
    if not row['is_fiction']:
        return 'non_fiction'
    # все остальное -- оригинальный македонский текст
    return 'original_mk'


# применяем классификацию к каждой строке
result_df['text_category'] = result_df.apply(classify_text, axis=1)

In [ ]:
# сохраняем полную классифицированную таблицу
classified_output = os.path.join(project_root, 'datasets', 'inventory_classified.csv')
result_df.to_csv(classified_output, index=False, encoding='utf-8-sig')
print(f'сохранено в {classified_output}')
print(f'всего столбцов: {len(result_df.columns)}')
print(f'столбцы: {list(result_df.columns)}')

# статистика по категориям
print('\nРаспределение по категориям текстов:')
category_counts = result_df['text_category'].value_counts()
for cat, count in category_counts.items():
    print(f'  {cat}: {count}')

# распределение по итоговому жанру
print('\nРаспределение по жанрам (genre_final):')
genre_counts = result_df['genre_final'].value_counts()
for genre, count in genre_counts.items():
    print(f'  {genre}: {count}')

# распределение по языку
print('\nРаспределение по языкам:')
lang_counts = result_df['language'].value_counts()
for lang, count in lang_counts.items():
    print(f'  {lang}: {count}')

In [ ]:
# фильтруем: оставляем только тексты для обучения нейросети
# критерии: язык mk, художественная литература, оригинал
filtered_df = result_df[result_df['text_category'] == 'original_mk'].copy()

print(f'Результат фильтрации:')
print(f'прошли фильтр: {len(filtered_df)} текстов')
print(f'исключены: {len(result_df) - len(filtered_df)} текстов')

# детали по исключенным текстам
excluded_df = result_df[result_df['text_category'] != 'original_mk']
print('\nисключены по причинам:')
for cat, count in excluded_df['text_category'].value_counts().items():
    print(f'  {cat}: {count}')

# жанры в отфильтрованном корпусе
print('\nЖанры в отфильтрованном корпусе:')
filtered_genre = filtered_df['genre_final'].value_counts()
for genre, count in filtered_genre.items():
    print(f'  {genre}: {count}')

# сохраняем отфильтрованный список файлов
filtered_output = os.path.join(project_root, 'datasets', 'corpus_filtered.csv')
filtered_df.to_csv(filtered_output, index=False, encoding='utf-8-sig')
print(f'\nотфильтрованный корпус сохранен в {filtered_output}')

# выводим список исключенных файлов для прозрачности
print('\nИсключенные файлы:')
for _, row in excluded_df.iterrows():
    print(f'  [{row["text_category"]}] {row["file_number"]}: {row["file_name"]} ({row["genre_final"]})')

## Резюме: результаты фильтрации корпуса

После определения языка и жанра для каждого из 128 файлов применили фильтрацию.
Для обучения нейросети оставили только оригинальные македонские художественные тексты.

**Отфильтрованный корпус: 98 текстов**
- Поэзия: 45
- Проза: 40
- Драма: 9
- Детская литература: 4

**Исключено: 30 текстов**
- Албанские тексты: 19
- Турецкие тексты: 6 (неожиданная находка -- в каталоге они не были отмечены как турецкие)
- Критика/эссеистика: 5

Результаты сохранены в:
- `datasets/processed/inventory_classified.csv` -- полная таблица со всеми столбцами (язык, жанр, категория)
- `datasets/processed/corpus_filtered.csv` -- только 98 текстов, прошедших фильтр

## 1.7. Чтение и очистка текстов

Читаем 98 отфильтрованных файлов из корпуса, очищаем от артефактов форматирования (BOM-маркер, лишние пробелы, управляющие символы, множественные пустые строки). Для каждого текста считаем базовую статистику: количество слов, символов, строк, абзацев. Очищенные тексты сохраняем в отдельную папку и в сводный CSV.

In [ ]:
# путь к папке с текстами
TEXTS_DIR = Path(r"C:/Projects/makedonian-course/texts")

# путь к CSV с отфильтрованным корпусом (результат предыдущего этапа)
CORPUS_CSV = Path(r"C:/Projects/makedonian-course/datasets/processed/corpus_filtered.csv")

# папка для очищенных текстов (отдельные файлы)
CLEANED_DIR = Path(r"C:/Projects/makedonian-course/datasets/cleaned_texts")

# путь для сводного CSV с текстами
OUTPUT_CSV = Path(r"C:/Projects/makedonian-course/datasets/raw/corpus_texts.csv")

In [ ]:
# функция чтения одного файла
def read_text_file(file_path):
    """
    читаем текстовый файл с encoding UTF-8-SIG (автоматически снимает BOM)
    возвращаем содержимое как строку, или None при ошибке
    """
    try:
        # encoding='utf-8-sig' убирает BOM-маркер при чтении
        with open(file_path, "r", encoding="utf-8-sig") as f:
            text = f.read()
        return text
    except UnicodeDecodeError as e:
        # если UTF-8-SIG не подошёл, пробуем обычный UTF-8
        print(f"  ошибка кодировки {file_path.name}: {e}")
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                text = f.read()
            return text
        except Exception as e2:
            print(f"  повторная ошибка {file_path.name}: {e2}")
            return None
    except Exception as e:
        # ловим любые другие ошибки (файл не найден, права доступа и т.д.)
        print(f"  ошибка чтения {file_path.name}: {e}")
        return None

In [ ]:
# функция очистки текста от артефактов
def clean_text(text):
    """
    очищаем текст от типичных артефактов форматирования:
    - BOM-маркер (если остался)
    - CRLF -> LF
    - управляющие символы (кроме перевода строки и табуляции)
    - пробелы/табы в конце каждой строки
    - множественные пустые строки подряд (оставляем максимум одну)
    - пустые строки в начале и конце файла
    """
    # убираем BOM, если он остался после чтения (символ \ufeff)
    text = text.lstrip("\ufeff")

    # нормализуем переносы строк: CRLF -> LF
    text = text.replace("\r\n", "\n")
    # на случай, если где-то одиночный \r (старый Mac-формат)
    text = text.replace("\r", "\n")

    # убираем управляющие символы (кроме \n и \t)
    # \x00-\x08 -- нулевой и прочие control-символы до табуляции
    # \x0b-\x0c -- vertical tab и form feed
    # \x0e-\x1f -- остальные control-символы
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", "", text)

    # разбиваем на строки, чтобы почистить каждую
    lines = text.split("\n")

    # убираем пробелы и табуляцию в конце каждой строки
    lines = [line.rstrip() for line in lines]

    # схлопываем множественные пустые строки подряд (оставляем максимум одну)
    cleaned_lines = []
    prev_empty = False
    for line in lines:
        # строка пустая?
        is_empty = len(line.strip()) == 0
        if is_empty and prev_empty:
            # пропускаем вторую (и далее) пустую строку подряд
            continue
        cleaned_lines.append(line)
        prev_empty = is_empty

    # убираем пустые строки в начале файла
    while cleaned_lines and cleaned_lines[0].strip() == "":
        cleaned_lines.pop(0)

    # убираем пустые строки в конце файла
    while cleaned_lines and cleaned_lines[-1].strip() == "":
        cleaned_lines.pop()

    # собираем обратно в текст
    text = "\n".join(cleaned_lines)

    return text

In [ ]:
# извлечение метаданных из первых строк текста
def extract_metadata_from_text(text):
    """
    многие файлы начинаются с имени автора и названия произведения
    пытаемся извлечь их из первых непустых строк

    возвращаем словарь с ключами:
    - 'author_in_text': автор из текста (или пустая строка)
    - 'title_in_text': название из текста (или пустая строка)
    - 'header_lines': сколько строк заняла шапка (автор + название)
    """
    # берем первые 10 непустых строк для анализа
    lines = text.split("\n")
    non_empty = []
    for line in lines[:30]:
        stripped = line.strip()
        if stripped:
            non_empty.append(stripped)
        if len(non_empty) >= 5:
            break

    result = {"author_in_text": "", "title_in_text": "", "header_lines": 0}

    if len(non_empty) < 1:
        return result

    # типичный паттерн: первая строка = автор (смешанный регистр), вторая = название (ВЕРХНИЙ РЕГИСТР)
    first = non_empty[0]
    second = non_empty[1] if len(non_empty) > 1 else ""

    # если первая строка выглядит как имя автора (кириллица, не слишком длинная)
    # а вторая -- как название (часто в верхнем регистре или просто короткая)
    is_cyrillic_name = bool(re.match(r"^[\u0410-\u042f\u0430-\u044f\u0400-\u04ff\s\.\-]+$", first)) and len(first) < 80
    is_latin_name = bool(re.match(r"^[A-Za-z\u00c0-\u00ff\s\.\-]+$", first)) and len(first) < 80

    if is_cyrillic_name or is_latin_name:
        result["author_in_text"] = first
        result["header_lines"] = 1

        # проверяем вторую строку как возможное название
        if second and len(second) < 120:
            result["title_in_text"] = second
            result["header_lines"] = 2

    # альтернативный паттерн: первая строка -- название в верхнем регистре
    elif first.isupper() and len(first) < 80:
        result["title_in_text"] = first
        result["header_lines"] = 1

    return result

In [ ]:
# подсчёт базовой статистики текста
def compute_text_stats(text):
    """
    считаем базовые метрики текста:
    - char_count: количество символов без пробелов
    - word_count: количество слов (разделение по пробелам)
    - line_count: количество строк
    - paragraph_count: количество абзацев (группы строк, разделенные пустыми строками)
    """
    # символы без пробелов и переводов строк
    char_count = len(text.replace(" ", "").replace("\n", "").replace("\t", ""))

    # слова: разбиваем по пробелам и переводам строк, фильтруем пустые
    words = text.split()
    word_count = len(words)

    # строки: считаем через split по \n
    lines = text.split("\n")
    line_count = len(lines)

    # абзацы: группы непустых строк, разделённые пустыми строками
    paragraph_count = 0
    in_paragraph = False
    for line in lines:
        if line.strip():
            if not in_paragraph:
                # начался новый абзац
                paragraph_count += 1
                in_paragraph = True
        else:
            # пустая строка -- конец абзаца
            in_paragraph = False

    return {
        "char_count": char_count,
        "word_count": word_count,
        "line_count": line_count,
        "paragraph_count": paragraph_count,
    }

In [ ]:
# загружаем таблицу отфильтрованного корпуса (98 текстов)
print("загружаем corpus_filtered.csv...")
corpus_df = pd.read_csv(CORPUS_CSV)
print(f"в таблице {len(corpus_df)} текстов")
print(f"жанры: {corpus_df['genre_final'].value_counts().to_dict()}")
print()

# создаём папку для очищенных текстов, если её нет
CLEANED_DIR.mkdir(parents=True, exist_ok=True)
print(f"папка для очищенных текстов: {CLEANED_DIR}")
print()

# основной цикл: читаем, чистим, считаем статистику
results = []
errors = []

for idx, row in corpus_df.iterrows():
    file_name = row["file_name"]
    file_number = row["file_number"]
    file_path = TEXTS_DIR / file_name

    # проверяем, что файл реально на диске
    if not file_path.exists():
        print(f"  файл не найден: {file_name}")
        errors.append({"file_number": file_number, "file_name": file_name, "error": "file not found"})
        continue

    # читаем файл
    raw_text = read_text_file(file_path)
    if raw_text is None:
        errors.append({"file_number": file_number, "file_name": file_name, "error": "read error"})
        continue

    # очищаем от артефактов
    cleaned = clean_text(raw_text)

    # пытаемся извлечь автора/название из первых строк
    meta = extract_metadata_from_text(cleaned)

    # считаем статистику
    stats = compute_text_stats(cleaned)

    # сохраняем очищенный текст как отдельный файл
    clean_file_name = f"{file_number}.txt"
    clean_file_path = CLEANED_DIR / clean_file_name
    with open(clean_file_path, "w", encoding="utf-8") as f:
        f.write(cleaned)

    # собираем строку для итогового DataFrame
    # маппинг столбцов: author <- author_from_catalog, title <- title_from_catalog,
    # genre <- genre_final, year <- year_clean
    result_row = {
        "file_number": file_number,
        "author": row["author_from_catalog"],
        "title": row["title_from_catalog"],
        "genre": row["genre_final"],
        "year": row["year_clean"] if pd.notna(row["year_clean"]) else None,
        "text": cleaned,
        "char_count": stats["char_count"],
        "word_count": stats["word_count"],
        "line_count": stats["line_count"],
        "paragraph_count": stats["paragraph_count"],
        "author_in_text": meta["author_in_text"],
        "title_in_text": meta["title_in_text"],
        "header_lines": meta["header_lines"],
    }
    results.append(result_row)

# выводим прогресс
print(f"обработано файлов: {len(results)}")
if errors:
    print(f"ошибки: {len(errors)}")
    for err in errors:
        print(f"  {err['file_number']}: {err['error']}")
print()

In [ ]:
# собираем итоговый DataFrame
texts_df = pd.DataFrame(results)

# сохраняем сводный CSV
# в CSV не включаем вспомогательные столбцы author_in_text, title_in_text, header_lines
output_columns = ["file_number", "author", "title", "genre", "year", "text", "char_count", "word_count"]
texts_df[output_columns].to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"сводный CSV сохранён: {OUTPUT_CSV}")
print(f"столбцы: {output_columns}")
print()

# выводим статистическую сводку
print("итоговая статистика корпуса:")
print(f"  текстов: {len(texts_df)}")
print(f"  общее количество слов: {texts_df['word_count'].sum():,}")
print(f"  общее количество символов (без пробелов): {texts_df['char_count'].sum():,}")
print(f"  среднее слов в тексте: {texts_df['word_count'].mean():,.0f}")
print(f"  медиана слов: {texts_df['word_count'].median():,.0f}")
print(f"  минимум слов: {texts_df['word_count'].min():,} ({texts_df.loc[texts_df['word_count'].idxmin(), 'title']})")
print(f"  максимум слов: {texts_df['word_count'].max():,} ({texts_df.loc[texts_df['word_count'].idxmax(), 'title']})")
print()

# статистика по жанрам
print("статистика по жанрам:")
genre_stats = texts_df.groupby("genre").agg(
    texts=("word_count", "count"),
    total_words=("word_count", "sum"),
    avg_words=("word_count", "mean"),
).reset_index()
for _, g in genre_stats.iterrows():
    print(f"  {g['genre']}: {g['texts']} текстов, {g['total_words']:,} слов (среднее {g['avg_words']:,.0f})")
print()

In [ ]:
# сравнение метаданных из первых строк с каталогом
print("сравнение метаданных из первых строк с каталогом:")
has_author_in_text = texts_df["author_in_text"].astype(bool).sum()
has_title_in_text = texts_df["title_in_text"].astype(bool).sum()
print(f"  автор найден в тексте: {has_author_in_text} из {len(texts_df)}")
print(f"  название найдено в тексте: {has_title_in_text} из {len(texts_df)}")
print()

# выводим первые 15 примеров, где нашли автора/название в тексте
print("примеры найденных метаданных (первые 15):")
found = texts_df[texts_df["author_in_text"] != ""].head(15)
for _, row in found.iterrows():
    print(f"  #{row['file_number']}:")
    print(f"    каталог:  {row['author']} — {row['title']}")
    print(f"    из текста: {row['author_in_text']} — {row['title_in_text']}")
print()

# считаем, у скольких файлов год не заполнен
year_na = texts_df["year"].isna().sum()
print(f"файлов без года издания: {year_na} из {len(texts_df)} (это нормально, NaN допустим)")
print()

# сводка по очищенным файлам
print(f"очищенные файлы сохранены в: {CLEANED_DIR}")
clean_count = len(list(CLEANED_DIR.glob("*.txt")))
print(f"  файлов в папке: {clean_count}")
print()

print("шаг 5 (чтение и очистка) завершён")

## 1.8. Оценка достаточности корпуса

Считаем общий объём корпуса в словах, оцениваем достаточность по порогу 500K слов (минимум для обучения генеративной модели). Также смотрим распределение по жанрам, авторам и годам. Проверяем доступные внешние корпуса македонского языка (Leipzig, OSCAR, Wikisource, MANU) и решаем, нужны ли дополнительные данные.

In [ ]:
# путь к корпусу, собранному на предыдущих шагах
CORPUS_PATH = Path(r"C:/Projects/makedonian-course/datasets/raw/corpus_texts.csv")

# порог достаточности корпуса в словах (из литературы по генеративным моделям)
THRESHOLD_WORDS = 500_000


def load_corpus(path):
    # читаем CSV с encoding utf-8-sig, чтобы убрать BOM-маркер
    df = pd.read_csv(path, encoding='utf-8-sig')
    return df

In [ ]:
def section_corpus_volume(df):
    # общая статистика по корпусу
    print()
    print('6.1 Оценка объема корпуса')
    print()

    # общее число текстов
    total_texts = len(df)
    # суммарное число слов во всем корпусе
    total_words = df['word_count'].sum()
    # суммарное число символов
    total_chars = df['char_count'].sum()
    # среднее число слов на текст
    avg_words = total_words / total_texts
    # медианное число слов (показывает "типичный" текст)
    median_words = df['word_count'].median()
    # самый длинный текст
    max_words = df['word_count'].max()
    # самый короткий текст
    min_words = df['word_count'].min()

    print(f'Текстов в корпусе:       {total_texts}')
    print(f'Всего слов:              {total_words:,}')
    print(f'Всего символов:          {total_chars:,}')
    print(f'Среднее слов на текст:   {avg_words:,.0f}')
    print(f'Медиана слов на текст:   {median_words:,.0f}')
    print(f'Минимум слов в тексте:   {min_words:,}')
    print(f'Максимум слов в тексте:  {max_words:,}')

    # отношение корпуса к порогу достаточности
    ratio = total_words / THRESHOLD_WORDS
    print(f'\nПорог достаточности:     {THRESHOLD_WORDS:,} слов')
    print(f'Корпус / порог:          {ratio:.1f}x')
    print(f'Корпус превышает порог в {ratio:.1f} раз')

    return total_words

In [ ]:
def section_genre_breakdown(df):
    # разбивка по жанрам
    print()
    print('6.1.1 Распределение по жанрам')
    print()

    # группируем по жанру, считаем количество текстов и сумму слов
    genre_stats = df.groupby('genre').agg(
        texts=('word_count', 'count'),
        total_words=('word_count', 'sum'),
        avg_words=('word_count', 'mean'),
        min_words=('word_count', 'min'),
        max_words=('word_count', 'max')
    ).sort_values('total_words', ascending=False)

    # общее число слов для расчета долей
    total_words = df['word_count'].sum()

    print(f'{"Жанр":<25} {"Текстов":>8} {"Всего слов":>15} {"Доля":>8} {"Средн. слов":>13}')
    print(f'{"-" * 25} {"-" * 8} {"-" * 15} {"-" * 8} {"-" * 13}')

    for genre, row in genre_stats.iterrows():
        # доля жанра от общего числа слов в процентах
        share = row['total_words'] / total_words * 100
        print(f'{genre:<25} {int(row["texts"]):>8} {int(row["total_words"]):>15,} {share:>7.1f}% {int(row["avg_words"]):>13,}')

    print(f'{"-" * 25} {"-" * 8} {"-" * 15} {"-" * 8} {"-" * 13}')
    print(f'{"ИТОГО":<25} {len(df):>8} {total_words:>15,} {"100.0%":>8}')

In [ ]:
def section_author_stats(df):
    # статистика по авторам (топ-20 по числу слов)
    print()
    print('6.1.2 Статистика по авторам (топ-20 по объему)')
    print()

    # группируем по автору: считаем тексты, сумму и среднее слов
    author_stats = df.groupby('author').agg(
        texts=('word_count', 'count'),
        total_words=('word_count', 'sum'),
        avg_words=('word_count', 'mean')
    ).sort_values('total_words', ascending=False)

    # общее число слов для расчета долей
    total_words = df['word_count'].sum()

    # сколько уникальных авторов в корпусе
    total_authors = len(author_stats)
    print(f'Всего авторов в корпусе: {total_authors}')
    print()

    print(f'{"#":>3} {"Автор":<30} {"Текстов":>8} {"Всего слов":>15} {"Доля":>8} {"Средн. слов":>13}')
    print(f'{"":>3} {"-" * 30} {"-" * 8} {"-" * 15} {"-" * 8} {"-" * 13}')

    # берем только топ-20 авторов
    for i, (author, row) in enumerate(author_stats.head(20).iterrows(), 1):
        # доля автора от общего объема
        share = row['total_words'] / total_words * 100
        # обрезаем имя автора, если слишком длинное
        name = author[:28] if len(author) > 28 else author
        print(f'{i:>3} {name:<30} {int(row["texts"]):>8} {int(row["total_words"]):>15,} {share:>7.1f}% {int(row["avg_words"]):>13,}')

    # сумма по остальным авторам
    remaining = author_stats.iloc[20:]
    if len(remaining) > 0:
        rem_texts = int(remaining['texts'].sum())
        rem_words = int(remaining['total_words'].sum())
        rem_share = rem_words / total_words * 100
        print(f'{"":>3} {"... остальные " + str(len(remaining)) + " авторов":<30} {rem_texts:>8} {rem_words:>15,} {rem_share:>7.1f}%')

In [ ]:
def section_year_distribution(df):
    # распределение по годам (у кого заполнен год)
    print()
    print('6.1.3 Распределение по годам издания')
    print()

    # сколько текстов имеют заполненный год
    has_year = df['year'].notna()
    print(f'Текстов с указанным годом:  {has_year.sum()} из {len(df)}')
    print(f'Текстов без года:           {(~has_year).sum()} из {len(df)}')

    if has_year.sum() > 0:
        # берем только тексты с годом
        df_year = df[has_year].copy()
        # переводим год в целое число
        df_year['year_int'] = df_year['year'].astype(int)
        # диапазон годов
        print(f'Диапазон годов:             {df_year["year_int"].min()} - {df_year["year_int"].max()}')

        # группируем по десятилетиям
        df_year['decade'] = (df_year['year_int'] // 10) * 10
        decade_stats = df_year.groupby('decade').agg(
            texts=('word_count', 'count'),
            total_words=('word_count', 'sum')
        ).sort_index()

        print(f'\nРаспределение по десятилетиям:')
        print(f'{"Десятилетие":<15} {"Текстов":>8} {"Слов":>15}')
        print(f'{"-" * 15} {"-" * 8} {"-" * 15}')
        for decade, row in decade_stats.iterrows():
            print(f'{decade}s{"":<10} {int(row["texts"]):>8} {int(row["total_words"]):>15,}')

In [ ]:
def section_leipzig(df):
    # оценка Leipzig Corpora Collection
    print()
    print('6.2 Оценка Leipzig Corpora Collection')
    print()

    print('Источник: Leipzig Corpora Collection (Universitat Leipzig)')
    print('Файл:    artifacts/Macedonian_sentences_Leipzig_Corpora_Collection (1).ipynb')
    print('Набор:   mkd_news_2020_100K (100 000 предложений на македонском)')
    print()
    print('Характеристика набора:')
    print('  - жанр: новостные тексты (news)')
    print('  - год: 2020')
    print('  - объем: ~100K предложений')
    print('  - HuggingFace: imvladikon/leipzig_corpora_collection')
    print()
    print('Оценка пригодности для проекта:')
    print()
    print('  1) Для обучения генерации юмора:  НЕ ПОДХОДИТ')
    print('     Причина: новостные тексты написаны в формальном стиле,')
    print('     без юмора, метафор и художественных приемов. Модель,')
    print('     обученная на новостях, будет генерировать формальный текст,')
    print('     а не юмористический.')
    print()
    print('  2) Для обучения лемматизатора:    ВОЗМОЖНО')
    print('     Новостные тексты содержат стандартную лексику и грамматику')
    print('     македонского языка. Если лемматизатор CLASSLA покажет')
    print('     недостаточное качество, новостной корпус можно использовать')
    print('     для дообучения.')
    print()
    print('  3) Для пополнения обучающей выборки: НЕ НУЖНО')
    print('     Наш литературный корпус уже содержит 4.6M слов, что')
    print('     достаточно для обучения. Смешивание жанров только')
    print('     ухудшит качество генерации юмора.')
    print()
    print('ВЫВОД: Leipzig Corpora mkd_news_2020_100K не включаем в обучающий')
    print('       корпус. Оставляем как запасной вариант для лемматизации.')

In [ ]:
def section_other_corpora():
    # обзор других открытых корпусов македонского языка
    print()
    print('6.3 Обзор других открытых корпусов македонского языка')
    print()

    # OSCAR
    print('1) OSCAR (Open Super-large Crawled Aggregated coRpus)')
    print('   URL:       https://oscar-project.org/')
    print('   Наличие:   Есть македонский сегмент (mk)')
    print('   Тип:       Web crawl (автоматический сбор со всего интернета)')
    print('   Оценка:    НЕ ПОДХОДИТ для обучения')
    print('   Причина:   Web crawl содержит смесь жанров: реклама, форумы,')
    print('              техническая документация, копипаст. Нет контроля')
    print('              над качеством и жанром текстов. Для генерации')
    print('              литературного юмора нужен чистый художественный')
    print('              корпус, а не случайный набор страниц из интернета.')
    print()

    # Wikisource
    print('2) Wikisource (Македонски Викиизвор)')
    print('   URL:       https://mk.wikisource.org/')
    print('   Наличие:   Есть, но крайне ограниченный контент')
    print('   Тип:       Литературные тексты (оригиналы и переводы)')
    print('   Оценка:    ТЕОРЕТИЧЕСКИ ИНТЕРЕСНО, но малый объем')
    print('   Причина:   Македонский Wikisource содержит очень мало текстов')
    print('              по сравнению с крупными языками. Большая часть')
    print('              контента -- переводы, а не оригинальные произведения.')
    print('              При нашем объеме корпуса (4.6M слов) добавление')
    print('              нескольких текстов из Wikisource не даст эффекта.')
    print()

    # Project Gutenberg
    print('3) Project Gutenberg')
    print('   URL:       https://www.gutenberg.org/')
    print('   Наличие:   Македонского раздела практически нет')
    print('   Тип:       Книги в общественном достоянии')
    print('   Оценка:    НЕ ПРИМЕНИМО')
    print('   Причина:   Project Gutenberg ориентирован на английский язык')
    print('              и крупные европейские языки. Македонских текстов')
    print('              в каталоге единицы (если вообще есть). Не стоит')
    print('              тратить время на поиск.')
    print()

    # MANU
    print('4) MANU (Македонска академија на науките и уметностите)')
    print('   Полное название: Corpus of the Macedonian Language (MANU)')
    print('   Тип:       Академический национальный корпус')
    print('   Оценка:    ИНТЕРЕСЕН, но недоступен для скачивания')
    print('   Причина:   Национальный корпус МАНУ содержит тексты разных')
    print('              жанров, включая литературные. Но доступ к корпусу')
    print('              ограничен: обычно предоставляется только через')
    print('              веб-интерфейс для конкорданс-запросов, а не')
    print('              для скачивания. Для нашего проекта не подходит:')
    print('              а) нет возможности скачать тексты целиком;')
    print('              б) наш корпус уже достаточен по объему.')
    print()

    print('ОБЩИЙ ВЫВОД ПО ВНЕШНИМ ИСТОЧНИКАМ:')
    print('  Ни один из рассмотренных открытых корпусов не дает преимуществ')
    print('  перед нашим собранным корпусом из 98 художественных текстов.')
    print('  Основные причины:')
    print('  - наш корпус уже превышает порог достаточности в 9 раз;')
    print('  - внешние корпуса содержат не литературные тексты (новости, веб);')
    print('  - литературные внешние корпуса (Wikisource, MANU) либо слишком')
    print('    малы, либо недоступны для скачивания.')

In [ ]:
def section_decision(total_words):
    # итоговое решение о достаточности корпуса
    print()
    print('6.4 Итоговое решение о достаточности корпуса')
    print()

    # отношение к порогу
    ratio = total_words / THRESHOLD_WORDS

    print(f'Собранный корпус:')
    print(f'  - 98 художественных текстов на македонском языке')
    print(f'  - 4 жанра: проза, поэзия, драма, детская книжевность')
    print(f'  - {total_words:,} слов (суммарно)')
    print()
    print(f'Порог достаточности для генеративной модели:')
    print(f'  - 500,000 слов (минимум для обучения character-level / word-level модели)')
    print(f'  - основан на опыте обучения LSTM/Transformer моделей на малых языках')
    print()
    print(f'Сравнение:')
    print(f'  - {total_words:,} / {THRESHOLD_WORDS:,} = {ratio:.1f}x')
    print(f'  - корпус превышает порог в {ratio:.1f} раз')
    print()
    print(f'РЕШЕНИЕ: дополнительные данные НЕ ТРЕБУЮТСЯ')
    print()
    print(f'Обоснование:')
    print(f'  1. Объем корпуса ({total_words:,} слов) многократно превышает')
    print(f'     минимальный порог (500,000 слов).')
    print(f'  2. Корпус состоит из оригинальных македонских художественных')
    print(f'     текстов, что соответствует задаче генерации юмора.')
    print(f'  3. Корпус разнообразен по жанрам (проза, поэзия, драма,')
    print(f'     детская литература) и авторам (несколько десятков).')
    print(f'  4. Внешние корпуса (Leipzig, OSCAR, Wikisource, MANU) не дают')
    print(f'     преимуществ: либо не тот жанр, либо недоступны, либо')
    print(f'     слишком малы.')
    print(f'  5. Смешивание корпусов из разных жанров ухудшит качество')
    print(f'     генерации художественного юмора.')
    print()
    print(f'Шаг 6.5 (скрипт загрузки): НЕ НУЖЕН, загрузка дополнительных')
    print(f'данных не требуется.')

In [ ]:
# запускаем анализ достаточности корпуса
print('Шаг 6: Анализ достаточности корпуса и обзор внешних источников')

# загружаем корпус
df = load_corpus(CORPUS_PATH)
print(f'Корпус загружен: {len(df)} текстов из {CORPUS_PATH}')

# 6.1 общая статистика
total_words = section_corpus_volume(df)

# 6.1.1 по жанрам
section_genre_breakdown(df)

# 6.1.2 по авторам
section_author_stats(df)

# 6.1.3 по годам
section_year_distribution(df)

# 6.2 Leipzig Corpora
section_leipzig(df)

# 6.3 другие открытые корпуса
section_other_corpora()

# 6.4 итоговое решение
section_decision(total_words)

## 1.9. Формирование финальных датасетов

Расширяем таблицу с текстами до полного датасета: добавляем столбцы id, source, is_original_mk через merge с corpus_filtered.csv. Разбиваем тексты на абзацы и сохраняем как отдельную таблицу. Создаём лёгкую таблицу метаданных (без тяжёлого столбца text) для быстрой загрузки.

In [ ]:
# путь к папке с датасетами
datasets_dir = 'C:/Projects/makedonian-course/datasets'

# путь к папке с очищенными текстами
cleaned_dir = os.path.join(datasets_dir, 'cleaned_texts')

In [ ]:
# 7.1 создаём corpus_full.csv
print('7.1 Создаем corpus_full.csv')
print()

# загружаем основной датасет с текстами (98 строк, 8 столбцов)
corpus_texts = pd.read_csv(
    os.path.join(datasets_dir, 'raw/corpus_texts.csv'),
    encoding='utf-8-sig'
)
print(f'  corpus_texts.csv: {len(corpus_texts)} строк, столбцы: {list(corpus_texts.columns)}')

# загружаем датасет с полными метаданными (98 строк, 27 столбцов)
corpus_filtered = pd.read_csv(
    os.path.join(datasets_dir, 'corpus_filtered.csv'),
    encoding='utf-8-sig'
)
print(f'  corpus_filtered.csv: {len(corpus_filtered)} строк, столбцы: {list(corpus_filtered.columns)}')

# из corpus_filtered берем только source и is_original_mk для merge
# file_number нужен как ключ для соединения
merge_cols = corpus_filtered[['file_number', 'source', 'is_original_mk']].copy()
print(f'  для merge выбрали столбцы: file_number, source, is_original_mk')

# соединяем corpus_texts с source и is_original_mk по file_number
corpus_full = corpus_texts.merge(merge_cols, on='file_number', how='left')
print(f'  после merge: {len(corpus_full)} строк')

# проверяем, что merge не потерял и не добавил строки
assert len(corpus_full) == 98, f'ожидали 98 строк, получили {len(corpus_full)}'

# добавляем колонку id (порядковый номер, начиная с 1)
corpus_full.insert(0, 'id', range(1, len(corpus_full) + 1))

# выстраиваем столбцы в нужном порядке
column_order = [
    'id', 'file_number', 'author', 'title', 'genre', 'year',
    'source', 'text', 'char_count', 'word_count', 'is_original_mk'
]
corpus_full = corpus_full[column_order]

# проверяем, что source подтянулся (не должно быть NaN)
missing_source = corpus_full['source'].isna().sum()
print(f'  пропущенных значений source: {missing_source}')

# проверяем is_original_mk
all_original = corpus_full['is_original_mk'].all()
print(f'  все тексты is_original_mk=True: {all_original}')

In [ ]:
# пересчитываем char_count и word_count по актуальному тексту
# char_count = количество непробельных символов (конвенция из предыдущих шагов)
# word_count = количество слов (split по пробелам)
# пересчёт нужен, потому что очистка могла немного изменить текст (схлопывание пустых строк)
old_chars = corpus_full['char_count'].copy()
old_words = corpus_full['word_count'].copy()
corpus_full['char_count'] = corpus_full['text'].apply(lambda t: len(''.join(t.split())))
corpus_full['word_count'] = corpus_full['text'].apply(lambda t: len(t.split()))
chars_changed = (old_chars != corpus_full['char_count']).sum()
words_changed = (old_words != corpus_full['word_count']).sum()
print(f'  пересчитан char_count: изменилось у {chars_changed} текстов')
print(f'  пересчитан word_count: изменилось у {words_changed} текстов')

# сохраняем corpus_full.csv
corpus_full_path = os.path.join(datasets_dir, 'raw/corpus_full.csv')
corpus_full.to_csv(corpus_full_path, index=False, encoding='utf-8-sig')
print(f'  сохранено: {corpus_full_path}')
print(f'  итого: {len(corpus_full)} текстов, {len(corpus_full.columns)} столбцов')
print()

In [ ]:
# 7.2 создаём corpus_paragraphs.csv (сегментация на абзацы)
print('7.2 Создаем corpus_paragraphs.csv (сегментация на абзацы)')
print()

# список для сбора всех абзацев
all_paragraphs = []

# глобальный счетчик id для абзацев
paragraph_id = 1

# проходим по каждому тексту в corpus_full
for _, row in corpus_full.iterrows():
    # text_id берем из столбца id в corpus_full
    text_id = row['id']

    # берем текст из столбца text
    full_text = row['text']

    # разбиваем текст по двойному переводу строки (абзацы/строфы)
    raw_paragraphs = full_text.split('\n\n')

    # номер абзаца внутри текста, начинаем с 1
    para_num = 0

    for para in raw_paragraphs:
        # убираем пробелы по краям
        para_stripped = para.strip()

        # пропускаем пустые абзацы
        if not para_stripped:
            continue

        # увеличиваем номер абзаца внутри текста
        para_num += 1

        # считаем слова в абзаце (split по пробелам)
        wc = len(para_stripped.split())

        # добавляем запись в список
        all_paragraphs.append({
            'id': paragraph_id,
            'text_id': text_id,
            'paragraph_number': para_num,
            'text': para_stripped,
            'word_count': wc
        })

        # увеличиваем глобальный id
        paragraph_id += 1

# собираем DataFrame из списка абзацев
paragraphs_df = pd.DataFrame(all_paragraphs)
print(f'  всего абзацев: {len(paragraphs_df)}')

# средние и медианные абзацы на текст
paras_per_text = paragraphs_df.groupby('text_id').size()
print(f'  среднее абзацев на текст: {paras_per_text.mean():.1f}')
print(f'  медиана абзацев на текст: {paras_per_text.median():.1f}')
print(f'  минимум абзацев в тексте: {paras_per_text.min()}')
print(f'  максимум абзацев в тексте: {paras_per_text.max()}')
print()

In [ ]:
# распределение абзацев по жанрам
# для этого подтягиваем жанр из corpus_full
genre_map = corpus_full.set_index('id')['genre']
paragraphs_with_genre = paragraphs_df.copy()
paragraphs_with_genre['genre'] = paragraphs_with_genre['text_id'].map(genre_map)

print('  распределение абзацев по жанрам:')
genre_para_stats = paragraphs_with_genre.groupby('genre').agg(
    texts=('text_id', 'nunique'),
    paragraphs=('id', 'count'),
    total_words=('word_count', 'sum'),
    avg_para_words=('word_count', 'mean')
).round(1)
print(genre_para_stats.to_string())
print()

# среднее количество слов в абзаце
avg_words = paragraphs_df['word_count'].mean()
print(f'  среднее слов в абзаце: {avg_words:.1f}')
print()

# сохраняем corpus_paragraphs.csv
paragraphs_path = os.path.join(datasets_dir, 'raw/corpus_paragraphs.csv')
paragraphs_df.to_csv(paragraphs_path, index=False, encoding='utf-8-sig')
print(f'  сохранено: {paragraphs_path}')
print()

In [ ]:
# 7.3 создаём corpus_metadata.csv и corpus_full.xlsx
print('7.3 Создаем corpus_metadata.csv и corpus_full.xlsx')
print()

# metadata = все столбцы кроме text (для быстрой загрузки без тяжелой колонки)
metadata_cols = [c for c in corpus_full.columns if c != 'text']
corpus_metadata = corpus_full[metadata_cols].copy()

# сохраняем corpus_metadata.csv
metadata_path = os.path.join(datasets_dir, 'corpus_metadata.csv')
corpus_metadata.to_csv(metadata_path, index=False, encoding='utf-8-sig')
print(f'  сохранено: {metadata_path}')
print(f'  столбцы: {list(corpus_metadata.columns)}')

# пробуем сохранить Excel-версию (нужен openpyxl)
try:
    import openpyxl  # noqa: F401
    xlsx_path = os.path.join(datasets_dir, 'corpus_full.xlsx')
    corpus_full.to_excel(xlsx_path, index=False, engine='openpyxl')
    print(f'  сохранено: {xlsx_path}')
except ImportError:
    print('  openpyxl не установлен, пропускаем сохранение в Excel')

print()

In [ ]:
# 7.4 проверка целостности
print('7.4 Проверка целостности')
print()

# проверяем, что все 98 file_number из corpus_filtered есть в corpus_full
filtered_fns = set(corpus_filtered['file_number'].tolist())
full_fns = set(corpus_full['file_number'].tolist())
missing_in_full = filtered_fns - full_fns
extra_in_full = full_fns - filtered_fns
print(f'  file_number из corpus_filtered, отсутствующие в corpus_full: {len(missing_in_full)}')
if missing_in_full:
    print(f'    пропущены: {missing_in_full}')
print(f'  file_number в corpus_full, отсутствующие в corpus_filtered: {len(extra_in_full)}')
if extra_in_full:
    print(f'    лишние: {extra_in_full}')

# проверяем, что нет пустых текстов
empty_texts = corpus_full[corpus_full['text'].isna() | (corpus_full['text'].str.strip() == '')]
print(f'  пустых текстов: {len(empty_texts)}')

# проверяем, что нет дублей по file_number
dupes = corpus_full['file_number'].duplicated().sum()
print(f'  дублей file_number: {dupes}')

# выборочная проверка char_count и word_count (первые 5 текстов)
print()
print('  выборочная проверка char_count и word_count (первые 5 текстов):')
for i in range(min(5, len(corpus_full))):
    row = corpus_full.iloc[i]
    # char_count = количество непробельных символов (конвенция из предыдущих шагов)
    actual_chars = len(''.join(row['text'].split()))
    actual_words = len(row['text'].split())
    chars_ok = actual_chars == row['char_count']
    words_ok = actual_words == row['word_count']
    status = 'OK' if (chars_ok and words_ok) else 'MISMATCH'
    print(f'    id={row["id"]}, file={row["file_number"]}: '
          f'chars={row["char_count"]} (actual={actual_chars}), '
          f'words={row["word_count"]} (actual={actual_words}) [{status}]')

# проверяем по всему корпусу, что char_count и word_count совпадают
all_chars_ok = True
all_words_ok = True
for _, row in corpus_full.iterrows():
    ac = len(''.join(row['text'].split()))
    aw = len(row['text'].split())
    if ac != row['char_count']:
        all_chars_ok = False
    if aw != row['word_count']:
        all_words_ok = False
print(f'  char_count корректен для всех 98 текстов: {all_chars_ok}')
print(f'  word_count корректен для всех 98 текстов: {all_words_ok}')

In [ ]:
# итоговая сводка
print()
print('ИТОГО')
print()

# общая статистика по корпусу
total_texts = len(corpus_full)
total_authors = corpus_full['author'].nunique()
total_genres = corpus_full['genre'].nunique()
total_words = corpus_full['word_count'].sum()
total_chars = corpus_full['char_count'].sum()
total_paragraphs = len(paragraphs_df)

print(f'  текстов: {total_texts}')
print(f'  авторов: {total_authors}')
print(f'  жанров: {total_genres} ({", ".join(corpus_full["genre"].unique())})')
print(f'  слов всего: {total_words:,}')
print(f'  символов всего: {total_chars:,}')
print(f'  абзацев всего: {total_paragraphs:,}')
print()

# распределение по жанрам (тексты)
print('  распределение текстов по жанрам:')
for genre, count in corpus_full['genre'].value_counts().items():
    words_in_genre = corpus_full[corpus_full['genre'] == genre]['word_count'].sum()
    print(f'    {genre}: {count} текстов, {words_in_genre:,} слов')
print()

# итоговые файлы
print('  созданные файлы:')
for fname in ['raw/corpus_full.csv', 'raw/corpus_paragraphs.csv', 'corpus_metadata.csv']:
    fpath = os.path.join(datasets_dir, fname)
    if os.path.exists(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        print(f'    {fname}: {size_kb:,.1f} KB')

# проверяем Excel
xlsx_path = os.path.join(datasets_dir, 'corpus_full.xlsx')
if os.path.exists(xlsx_path):
    size_kb = os.path.getsize(xlsx_path) / 1024
    print(f'    corpus_full.xlsx: {size_kb:,.1f} KB')

print()
print('Готово!')

## Резюме: структура финальных датасетов

На этом шаге сформированы три итоговых файла:

**corpus_full.csv** (98 текстов, 11 столбцов):
- `id` -- порядковый номер (1-98)
- `file_number` -- номер файла в исходной папке texts/
- `author` -- автор из каталога
- `title` -- название из каталога
- `genre` -- жанр (проза, поэзия, драма, детская книжевность)
- `year` -- год издания (NaN, если не известен)
- `source` -- источник текста
- `text` -- полный очищенный текст
- `char_count` -- количество непробельных символов
- `word_count` -- количество слов
- `is_original_mk` -- флаг оригинального македонского текста

**corpus_paragraphs.csv** (48,061 абзац, 5 столбцов):
- `id` -- глобальный id абзаца
- `text_id` -- ссылка на id в corpus_full
- `paragraph_number` -- номер абзаца внутри текста
- `text` -- текст абзаца
- `word_count` -- количество слов в абзаце

**corpus_metadata.csv** (98 строк, 10 столбцов):
- все столбцы из corpus_full кроме `text` -- лёгкая таблица для быстрой загрузки метаданных без тяжёлого текстового столбца

## 1.10. Разбиение на train / validation / test

Разбиваем корпус на три части: **train** (70%), **validation** (15%) и **test** (15%).

Стратегия разбиения:

- **Стратификация по жанру**: в каждой выборке должны быть представлены все жанры примерно в тех же пропорциях, что и в полном корпусе. Жанр "Детска книжевност" (всего 4 текста) объединяем с "Проза" для целей стратификации, потому что 4 текста слишком мало для разбиения на три части.

- **Группировка по авторам**: тексты одного автора попадают только в одну выборку. Если один автор окажется и в train, и в test, модель на test может показать завышенное качество за счёт запоминания авторского стиля.

- **Ручное распределение мультитекстных авторов**: у нас 3 автора с несколькими текстами:
  - Лужина Јелена -- 4 текста (Драма) -- в train
  - Јоциќ Светлана -- 3 текста (Поезија + 2 Детска книжевност) -- в train
  - Силјан Раде -- 2 текста (Проза + Поезија) -- в train

  Эти 9 текстов уходят в train вручную, остальные 89 авторов (у каждого по 1 тексту) распределяем через `train_test_split` с параметром `stratify`.

- **Абзацы следуют за текстами**: все абзацы одного текста попадают в ту же выборку, что и сам текст.

In [ ]:
# новые импорты для этой части (sklearn для стратифицированного разбиения)
from sklearn.model_selection import train_test_split

In [ ]:
# фиксируем seed для воспроизводимости
RANDOM_STATE = 42

# пропорции разбиения: 70% train, 15% validation, 15% test
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# загружаем полный корпус с текстами
corpus_full = pd.read_csv(os.path.join(DATASETS_DIR, 'raw/corpus_full.csv'))
print(f'Загружен corpus_full.csv: {len(corpus_full)} текстов, {corpus_full.shape[1]} столбцов')
print(f'Столбцы: {list(corpus_full.columns)}')
print()

# загружаем параграфы
corpus_paragraphs = pd.read_csv(os.path.join(DATASETS_DIR, 'raw/corpus_paragraphs.csv'))
print(f'Загружен corpus_paragraphs.csv: {len(corpus_paragraphs)} параграфов')
print()

# загружаем метаданные (без текстов) для удобной работы
metadata = pd.read_csv(os.path.join(DATASETS_DIR, 'corpus_metadata.csv'))
print(f'Загружен corpus_metadata.csv: {len(metadata)} текстов')

In [ ]:
# смотрим распределение жанров
print('Распределение жанров:')
genre_counts = metadata['genre'].value_counts()
for genre, count in genre_counts.items():
    print(f'  {genre}: {count}')
print()

# смотрим авторов с несколькими текстами
author_counts = metadata['author'].value_counts()
# отбираем авторов, у которых больше одного текста
multi_authors = author_counts[author_counts > 1]
print(f'Авторы с несколькими текстами ({len(multi_authors)} авторов):')
for author, count in multi_authors.items():
    # показываем жанры текстов этого автора
    genres = metadata[metadata['author'] == author]['genre'].tolist()
    print(f'  {author}: {count} текстов, жанры: {genres}')

In [ ]:
# создаем колонку для стратификации
# объединяем "Детска книжевност" с "Проза" -- оба жанра прозаические
metadata['genre_stratify'] = metadata['genre'].replace('Детска книжевност', 'Проза')
print('Для стратификации объединяем "Детска книжевност" с "Проза":')
stratify_counts = metadata['genre_stratify'].value_counts()
for genre, count in stratify_counts.items():
    print(f'  {genre}: {count}')
print()

# готовим колонку split
metadata['split'] = None

# Лужина Јелена: 4 текста, все Драма -> в train (нужно больше всего данных)
luzina_mask = metadata['author'] == 'Лужина Јелена'
metadata.loc[luzina_mask, 'split'] = 'train'
print('Лужина Јелена (4 текста, Драма) -> train')

# Јоциќ Светлана: 3 текста (1 Поезија + 2 Детска книжевност)
# отправляем в train -- тут 2 из 4 "Детска книжевност", важно для обучения
jocic_mask = metadata['author'] == 'Јоциќ Светлана'
metadata.loc[jocic_mask, 'split'] = 'train'
print('Јоциќ Светлана (3 текста: Поезија + 2 Детска книжевност) -> train')

# Силјан Раде: 2 текста (Проза + Поезија) -> в train
siljan_mask = metadata['author'] == 'Силјан Раде'
metadata.loc[siljan_mask, 'split'] = 'train'
print('Силјан Раде (2 текста: Проза + Поезија) -> train')
print()

# считаем, сколько текстов уже распределено
assigned_count = metadata['split'].notna().sum()
remaining_count = metadata['split'].isna().sum()
print(f'Вручную распределено: {assigned_count} текстов в train')
print(f'Осталось распределить: {remaining_count} текстов')

In [ ]:
# собираем оставшиеся тексты для стратифицированного разбиения
remaining = metadata[metadata['split'].isna()].copy()

# целевые размеры для val и test: по 15% от всех 98 текстов
# 98 * 0.15 = 14.7, округляем -- примерно 15 текстов в val и 15 в test
# уже назначено 9 в train, значит из 89 нужно взять ~15 в val, ~15 в test, ~59 в train
# итого train: 9 + 59 = 68, val: 15, test: 15 -- суммарно 98
val_test_size = round(98 * (VAL_RATIO + TEST_RATIO))
print(f'Целевой размер val+test: {val_test_size} текстов')

# разбиваем оставшиеся на train и (val+test)
remaining_train, remaining_valtest = train_test_split(
    remaining,
    test_size=val_test_size,
    random_state=RANDOM_STATE,
    stratify=remaining['genre_stratify']
)
print(f'Из оставшихся 89: {len(remaining_train)} в train, {len(remaining_valtest)} в val+test')

# теперь делим val+test пополам: 50/50
remaining_val, remaining_test = train_test_split(
    remaining_valtest,
    test_size=0.5,
    random_state=RANDOM_STATE,
    stratify=remaining_valtest['genre_stratify']
)
print(f'val+test разделено: {len(remaining_val)} в validation, {len(remaining_test)} в test')

In [ ]:
# записываем результаты в metadata
metadata.loc[remaining_train.index, 'split'] = 'train'
metadata.loc[remaining_val.index, 'split'] = 'validation'
metadata.loc[remaining_test.index, 'split'] = 'test'

# проверяем, что все тексты получили split
assert metadata['split'].isna().sum() == 0, 'Есть тексты без split!'

# итоговая статистика по split
print('Итоговое разбиение:')
split_counts = metadata['split'].value_counts()
for split_name in ['train', 'validation', 'test']:
    count = split_counts.get(split_name, 0)
    pct = count / len(metadata) * 100
    print(f'  {split_name}: {count} текстов ({pct:.1f}%)')
print()

# статистика по жанрам в каждом split
print('Жанры по split:')
for split_name in ['train', 'validation', 'test']:
    subset = metadata[metadata['split'] == split_name]
    genre_dist = subset['genre'].value_counts()
    print(f'  {split_name}:')
    for genre, count in genre_dist.items():
        total_genre = genre_counts[genre]
        pct = count / total_genre * 100
        print(f'    {genre}: {count} из {total_genre} ({pct:.1f}%)')
print()

# статистика по словам в каждом split
print('Количество слов по split:')
total_words = metadata['word_count'].sum()
for split_name in ['train', 'validation', 'test']:
    subset = metadata[metadata['split'] == split_name]
    words = subset['word_count'].sum()
    pct = words / total_words * 100
    print(f'  {split_name}: {words:,} слов ({pct:.1f}%)')
print(f'  всего: {total_words:,} слов')

In [ ]:
# проверяем, что авторы не пересекаются между split
print('Проверка: авторы не повторяются между split')
for split_a in ['train', 'validation', 'test']:
    for split_b in ['train', 'validation', 'test']:
        if split_a >= split_b:
            continue
        authors_a = set(metadata[metadata['split'] == split_a]['author'])
        authors_b = set(metadata[metadata['split'] == split_b]['author'])
        overlap = authors_a & authors_b
        if overlap:
            print(f'  ВНИМАНИЕ: авторы в {split_a} и {split_b} пересекаются: {overlap}')
        else:
            print(f'  {split_a} и {split_b}: пересечений нет')

In [ ]:
# назначаем split параграфам
# каждый параграф получает split от своего текста
text_to_split = metadata.set_index('id')['split'].to_dict()

# добавляем колонку split к параграфам через text_id
corpus_paragraphs['split'] = corpus_paragraphs['text_id'].map(text_to_split)

# проверяем, что все параграфы получили split
missing_split = corpus_paragraphs['split'].isna().sum()
if missing_split > 0:
    print(f'ВНИМАНИЕ: {missing_split} параграфов без split!')
else:
    print(f'Все {len(corpus_paragraphs)} параграфов получили split')
print()

# статистика параграфов по split
print('Параграфы по split:')
para_split_counts = corpus_paragraphs['split'].value_counts()
for split_name in ['train', 'validation', 'test']:
    count = para_split_counts.get(split_name, 0)
    pct = count / len(corpus_paragraphs) * 100
    # считаем слова в параграфах этого split
    words = corpus_paragraphs[corpus_paragraphs['split'] == split_name]['word_count'].sum()
    print(f'  {split_name}: {count:,} параграфов ({pct:.1f}%), {words:,} слов')

In [ ]:
# добавляем split к corpus_full
corpus_full['split'] = corpus_full['id'].map(text_to_split)

# сохраняем train.csv, validation.csv, test.csv (полные тексты с метаданными)
for split_name in ['train', 'validation', 'test']:
    split_df = corpus_full[corpus_full['split'] == split_name].copy()
    # train.csv больше 25 МБ, храним в raw/
    subdir = 'raw' if split_name == 'train' else ''
    filepath = os.path.join(DATASETS_DIR, subdir, f'{split_name}.csv') if subdir else os.path.join(DATASETS_DIR, f'{split_name}.csv')
    split_df.to_csv(filepath, index=False, encoding='utf-8')
    print(f'Сохранен {split_name}.csv: {len(split_df)} текстов')

# сохраняем split_info.csv -- какой текст в каком split
split_info = metadata[['id', 'file_number', 'author', 'title', 'genre', 'word_count', 'split']].copy()
split_info.to_csv(os.path.join(DATASETS_DIR, 'split_info.csv'), index=False, encoding='utf-8')
print(f'Сохранен split_info.csv: {len(split_info)} строк')

# обновляем corpus_paragraphs.csv с колонкой split
corpus_paragraphs.to_csv(os.path.join(DATASETS_DIR, 'raw/corpus_paragraphs.csv'), index=False, encoding='utf-8')
print(f'Обновлен corpus_paragraphs.csv: добавлена колонка split')

# сохраняем отдельные файлы параграфов по split
for split_name in ['train', 'validation', 'test']:
    split_paras = corpus_paragraphs[corpus_paragraphs['split'] == split_name].copy()
    # train_paragraphs.csv больше 25 МБ, храним в raw/
    subdir = 'raw' if split_name == 'train' else ''
    filepath = os.path.join(DATASETS_DIR, subdir, f'{split_name}_paragraphs.csv') if subdir else os.path.join(DATASETS_DIR, f'{split_name}_paragraphs.csv')
    split_paras.to_csv(filepath, index=False, encoding='utf-8')
    print(f'Сохранен {split_name}_paragraphs.csv: {len(split_paras)} параграфов')

In [ ]:
# визуализация: количество текстов в каждом split, жанры по split, слова по split
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# график 1: гистограмма текстов по split
ax1 = axes[0]
splits = ['train', 'validation', 'test']
counts = [split_counts.get(s, 0) for s in splits]
colors_split = ['#2196F3', '#FF9800', '#4CAF50']
bars = ax1.bar(splits, counts, color=colors_split, edgecolor='black', linewidth=0.5)
# подписываем значения над столбцами
for bar, count in zip(bars, counts):
    pct = count / len(metadata) * 100
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
             f'{count} ({pct:.0f}%)', ha='center', va='bottom', fontsize=10)
ax1.set_ylabel('Количество текстов')
ax1.set_title('Распределение текстов по split')
ax1.set_ylim(0, max(counts) * 1.15)

# график 2: жанры в каждом split (grouped bar chart)
ax2 = axes[1]
genres = ['Поезија', 'Проза', 'Драма', 'Детска книжевност']
# сокращаем названия для графика
genre_labels = ['Поезија', 'Проза', 'Драма', 'Детска\nкнижевност']
x = np.arange(len(genres))
width = 0.25

for i, split_name in enumerate(splits):
    subset = metadata[metadata['split'] == split_name]
    vals = [len(subset[subset['genre'] == g]) for g in genres]
    bars = ax2.bar(x + i * width, vals, width, label=split_name, color=colors_split[i],
                   edgecolor='black', linewidth=0.5)
    # подписываем значения
    for bar, val in zip(bars, vals):
        if val > 0:
            ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
                     str(val), ha='center', va='bottom', fontsize=9)

ax2.set_ylabel('Количество текстов')
ax2.set_title('Жанры по split')
ax2.set_xticks(x + width)
ax2.set_xticklabels(genre_labels)
ax2.legend()
ax2.set_ylim(0, ax2.get_ylim()[1] * 1.15)

# график 3: слова по split
ax3 = axes[2]
word_counts_by_split = []
for split_name in splits:
    subset = metadata[metadata['split'] == split_name]
    word_counts_by_split.append(subset['word_count'].sum())

bars = ax3.bar(splits, word_counts_by_split, color=colors_split, edgecolor='black', linewidth=0.5)
for bar, wc in zip(bars, word_counts_by_split):
    pct = wc / total_words * 100
    ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + total_words * 0.01,
             f'{wc:,}\n({pct:.0f}%)', ha='center', va='bottom', fontsize=9)
ax3.set_ylabel('Количество слов')
ax3.set_title('Слова по split')
ax3.set_ylim(0, max(word_counts_by_split) * 1.2)

plt.tight_layout()
plt.savefig(os.path.join(TEMP_DIR, 'step8_split_overview.png'), dpi=150, bbox_inches='tight')
print('Сохранена визуализация: step8_split_overview.png')
plt.show()
plt.close()

In [ ]:
# сводная таблица разбиения (визуализация в виде таблицы)
fig, ax = plt.subplots(figsize=(12, 6))
ax.axis('off')

# формируем данные для таблицы
table_data = []
for split_name in ['train', 'validation', 'test']:
    subset = metadata[metadata['split'] == split_name]
    n_texts = len(subset)
    n_words = subset['word_count'].sum()
    n_poetry = len(subset[subset['genre'] == 'Поезија'])
    n_prose = len(subset[subset['genre'] == 'Проза'])
    n_drama = len(subset[subset['genre'] == 'Драма'])
    n_kids = len(subset[subset['genre'] == 'Детска книжевност'])
    # параграфы
    n_paras = len(corpus_paragraphs[corpus_paragraphs['split'] == split_name])
    table_data.append([
        split_name, n_texts, f'{n_words:,}', n_poetry, n_prose, n_drama, n_kids, f'{n_paras:,}'
    ])

# добавляем строку "Всего"
table_data.append([
    'Всего', len(metadata), f'{total_words:,}',
    genre_counts.get('Поезија', 0), genre_counts.get('Проза', 0),
    genre_counts.get('Драма', 0), genre_counts.get('Детска книжевност', 0),
    f'{len(corpus_paragraphs):,}'
])

col_labels = ['Split', 'Текстов', 'Слов', 'Поезија', 'Проза', 'Драма', 'Детска кн.', 'Параграфов']

table = ax.table(
    cellText=table_data,
    colLabels=col_labels,
    loc='center',
    cellLoc='center'
)

# стилизуем заголовок таблицы
for j in range(len(col_labels)):
    table[0, j].set_facecolor('#37474F')
    table[0, j].set_text_props(color='white', fontsize=11, weight='bold')

# стилизуем строки
row_colors = ['#E3F2FD', '#FFF3E0', '#E8F5E9', '#F5F5F5']
for i in range(len(table_data)):
    for j in range(len(col_labels)):
        table[i + 1, j].set_facecolor(row_colors[i])
        table[i + 1, j].set_text_props(fontsize=10)

table.auto_set_font_size(False)
table.scale(1.2, 1.8)

ax.set_title('Сводная таблица разбиения корпуса', fontsize=14, fontweight='bold', pad=20)
plt.savefig(os.path.join(TEMP_DIR, 'step8_split_table.png'), dpi=150, bbox_inches='tight')
print('Сохранена визуализация: step8_split_table.png')
plt.show()
plt.close()

In [ ]:
# выводим список текстов в каждом split
print('Список текстов по split:')
for split_name in ['train', 'validation', 'test']:
    subset = metadata[metadata['split'] == split_name].sort_values('id')
    print(f'\n  {split_name.upper()} ({len(subset)} текстов):')
    for _, row in subset.iterrows():
        print(f'    id={row["id"]:2d}  {row["author"]:<30s}  {row["genre"]:<25s}  {row["title"][:50]}')

## 1.11. Разведочный анализ данных (EDA)

Строим визуализации, чтобы лучше понять корпус: как тексты распределены по жанрам, объёмам, авторам, годам. В конце строим облако слов -- самые частые слова корпуса (без стоп-слов).

In [ ]:
# новые импорты для EDA
from wordcloud import WordCloud
from collections import Counter
import matplotlib.ticker

# путь к шрифту для wordcloud (Arial поддерживает кириллицу)
FONT_PATH = 'C:/Windows/Fonts/arial.ttf'

# настраиваем matplotlib для кириллицы
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 12

In [ ]:
# загружаем метаданные и полный корпус заново для EDA
meta = pd.read_csv(os.path.join(DATASETS_DIR, 'corpus_metadata.csv'))
print(f'Загружено метаданных: {len(meta)} строк')

# загружаем полный корпус (с текстами) для word cloud
corpus = pd.read_csv(os.path.join(DATASETS_DIR, 'raw/corpus_full.csv'))
print(f'Загружено текстов: {len(corpus)} строк')

In [ ]:
# 9.1 Распределение по жанрам (pie chart)

# считаем количество текстов в каждом жанре
genre_counts_eda = meta['genre'].value_counts()
print('Распределение по жанрам:')
print(genre_counts_eda)
print()

# словарь перевода жанров на русский для подписей
genre_ru = {
    'Поезија': 'Поэзия',
    'Проза': 'Проза',
    'Драма': 'Драма',
    'Детска книжевност': 'Детская литература'
}

# подписи для pie chart: русский перевод + количество
labels = [f'{genre_ru.get(g, g)} ({c})' for g, c in zip(genre_counts_eda.index, genre_counts_eda.values)]

# цвета для жанров
colors_eda = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2']

# рисуем круговую диаграмму
fig, ax = plt.subplots(figsize=(8, 6))
wedges, texts, autotexts = ax.pie(
    genre_counts_eda.values,
    labels=labels,
    autopct='%1.1f%%',
    colors=colors_eda,
    startangle=90,
    textprops={'fontsize': 13}
)

# делаем проценты жирными
for t in autotexts:
    t.set_fontweight('bold')

ax.set_title('Распределение текстов по жанрам', fontsize=16, fontweight='bold')

plt.tight_layout()
fig.savefig(os.path.join(TEMP_DIR, 'eda_genre_pie.png'), dpi=150, bbox_inches='tight')
print('Сохранено: eda_genre_pie.png')
plt.show()
plt.close(fig)

In [ ]:
# 9.2 Распределение объема текстов (histogram)

# берем столбец word_count
word_counts_eda = meta['word_count']

# считаем среднее и медиану
mean_wc = word_counts_eda.mean()
median_wc = word_counts_eda.median()
print(f'Среднее количество слов: {mean_wc:,.0f}')
print(f'Медиана: {median_wc:,.0f}')
print(f'Минимум: {word_counts_eda.min():,}')
print(f'Максимум: {word_counts_eda.max():,}')
print()

# рисуем гистограмму
fig, ax = plt.subplots(figsize=(10, 6))

# 20 бинов для наглядности
ax.hist(word_counts_eda, bins=20, color='#4e79a7', edgecolor='white', alpha=0.85)

# вертикальная линия для среднего
ax.axvline(mean_wc, color='#e15759', linestyle='--', linewidth=2,
           label=f'Среднее: {mean_wc:,.0f}')

# вертикальная линия для медианы
ax.axvline(median_wc, color='#f28e2b', linestyle='-', linewidth=2,
           label=f'Медиана: {median_wc:,.0f}')

ax.set_xlabel('Количество слов в тексте', fontsize=13)
ax.set_ylabel('Количество текстов', fontsize=13)
ax.set_title('Распределение объема текстов (по числу слов)', fontsize=15, fontweight='bold')
ax.legend(fontsize=12)

plt.tight_layout()
fig.savefig(os.path.join(TEMP_DIR, 'eda_word_count_hist.png'), dpi=150, bbox_inches='tight')
print('Сохранено: eda_word_count_hist.png')
plt.show()
plt.close(fig)

In [ ]:
# 9.3 Топ-20 авторов по суммарному объему (bar chart)

# суммируем word_count по автору (у некоторых авторов несколько произведений)
author_total = meta.groupby('author')['word_count'].sum().sort_values(ascending=False)

# берем топ-20
top20_authors = author_total.head(20)
print('Топ-20 авторов по суммарному объему:')
print(top20_authors.to_string())
print()

# рисуем горизонтальный bar chart (чтобы имена читались)
fig, ax = plt.subplots(figsize=(10, 8))

# переворачиваем порядок, чтобы самый большой автор был сверху
top20_reversed = top20_authors.iloc[::-1]

ax.barh(top20_reversed.index, top20_reversed.values, color='#4e79a7', edgecolor='white')

ax.set_xlabel('Суммарное количество слов', fontsize=13)
ax.set_title('Топ-20 авторов по суммарному объему текстов', fontsize=15, fontweight='bold')

# добавляем числовые значения рядом с каждой полосой
for i, (author, val) in enumerate(zip(top20_reversed.index, top20_reversed.values)):
    ax.text(val + 1000, i, f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
fig.savefig(os.path.join(TEMP_DIR, 'eda_top20_authors.png'), dpi=150, bbox_inches='tight')
print('Сохранено: eda_top20_authors.png')
plt.show()
plt.close(fig)

In [ ]:
# 9.4 Хронология текстов (scatter plot по годам)

# отбираем тексты с заполненным годом
with_year = meta[meta['year'].notna()].copy()
without_year_count = meta['year'].isna().sum()
print(f'Текстов с годом: {len(with_year)}')
print(f'Текстов без года: {without_year_count}')
print(f'Диапазон годов: {int(with_year["year"].min())} -- {int(with_year["year"].max())}')
print()

# цвета для жанров
genre_colors = {
    'Поезија': '#4e79a7',
    'Проза': '#f28e2b',
    'Драма': '#e15759',
    'Детска книжевност': '#76b7b2'
}

# рисуем scatter plot: год vs word_count, цвет по жанру
fig, ax = plt.subplots(figsize=(12, 6))

# рисуем каждый жанр отдельно, чтобы легенда была по жанрам
for genre, color in genre_colors.items():
    # фильтруем тексты этого жанра
    subset = with_year[with_year['genre'] == genre]
    if len(subset) == 0:
        continue
    ax.scatter(
        subset['year'], subset['word_count'],
        c=color, s=80, alpha=0.8, edgecolors='white', linewidth=0.5,
        label=genre_ru.get(genre, genre)
    )

ax.set_xlabel('Год издания', fontsize=13)
ax.set_ylabel('Количество слов', fontsize=13)
ax.set_title(
    f'Хронология текстов (показаны {len(with_year)} из {len(meta)}, '
    f'{without_year_count} без года)',
    fontsize=14, fontweight='bold'
)
ax.legend(fontsize=11)

# показываем целые годы на оси X
ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))

plt.tight_layout()
fig.savefig(os.path.join(TEMP_DIR, 'eda_year_scatter.png'), dpi=150, bbox_inches='tight')
print('Сохранено: eda_year_scatter.png')
plt.show()
plt.close(fig)

In [ ]:
# 9.5 Облако слов (word cloud)

# список стоп-слов для македонского языка
# стандартные библиотеки не содержат македонских стоп-слов, поэтому составляем вручную
# включаем союзы, предлоги, частицы, местоимения, вспомогательные глаголы
MK_STOP_WORDS = {
    # союзы и частицы
    'и', 'на', 'во', 'за', 'од', 'со', 'не', 'да', 'се', 'е',
    'но', 'а', 'ни', 'ке', 'ќе', 'би', 'ли', 'ма', 'ми', 'му',
    # местоимения и местоименные формы
    'го', 'ги', 'ја', 'тој', 'таа', 'тоа', 'тие', 'ние', 'вие',
    'јас', 'ти', 'нас', 'вас', 'нив', 'нам', 'вам', 'ним',
    'што', 'ова', 'кој', 'која', 'кое', 'кои', 'тоа', 'овој', 'оваа',
    'тој', 'него', 'нему', 'нејзе', 'нив',
    'мој', 'моја', 'мое', 'мои', 'твој', 'твоја', 'твое', 'твои',
    'свој', 'своја', 'свое', 'свои', 'негов', 'негова', 'негово',
    'нејзин', 'нејзина', 'нејзино', 'наш', 'наша', 'наше', 'наши',
    'ваш', 'ваша', 'ваше', 'ваши', 'нивни', 'нивна', 'нивно',
    # предлоги и наречия
    'по', 'при', 'пред', 'без', 'меѓу', 'над', 'под', 'до', 'низ', 'кон',
    'како', 'кога', 'каде', 'зошто', 'колку', 'веќе', 'уште', 'само',
    'тука', 'таму', 'сега', 'тогаш', 'потоа', 'пак', 'ете', 'еве',
    'многу', 'малку', 'сите', 'секој', 'некој', 'ништо', 'нешто',
    'овде', 'онде', 'некаде',
    # вспомогательные глаголы и частицы
    'сум', 'си', 'беше', 'бил', 'била', 'било', 'биле', 'бев',
    'сме', 'сте', 'се', 'им', 'има', 'нема',
    'може', 'мора', 'треба',
    # числа и очень частые слова
    'еден', 'една', 'едно', 'два', 'две', 'три',
    'тоа', 'тие', 'ние', 'вие',
    # другие частые служебные слова
    'дека', 'ако', 'или', 'ама', 'иако', 'затоа', 'бидејќи',
    'нека', 'некои', 'него', 'неа', 'нив',
    'овие', 'оние', 'оној', 'онаа', 'оние',
    'сиот', 'сета', 'сето', 'секоја', 'секое',
    'така', 'толку', 'дури',
    'ден', 'ноќ',
}

print(f'Стоп-слов в списке: {len(MK_STOP_WORDS)}')

In [ ]:
# склеиваем все тексты корпуса в одну строку
all_text = ' '.join(corpus['text'].dropna().astype(str))

# очищаем текст: оставляем только буквы (кириллица + латиница) и пробелы
all_text_clean = re.sub(r'[^а-яА-ЯёЁa-zA-ZѓЃќЌљЉњЊџЏјЈѕЅ\s]', ' ', all_text)

# приводим к нижнему регистру
all_text_lower = all_text_clean.lower()

# разбиваем на слова
words = all_text_lower.split()

# фильтруем стоп-слова и слова короче 3 букв
filtered_words = [w for w in words if w not in MK_STOP_WORDS and len(w) >= 3]

print(f'Всего слов после очистки: {len(filtered_words):,}')

# считаем частоты слов
word_freq = Counter(filtered_words)

# топ-30 слов
print('Топ-30 слов:')
for word, count in word_freq.most_common(30):
    print(f'  {word}: {count}')

In [ ]:
# строим word cloud
wc = WordCloud(
    width=1200,
    height=600,
    max_words=200,
    background_color='white',
    font_path=FONT_PATH,
    random_state=42,
    colormap='viridis',
    collocations=False
)

# генерируем облако из частот
wc.generate_from_frequencies(word_freq)

# сохраняем как картинку
fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title('Облако слов корпуса (без стоп-слов)', fontsize=16, fontweight='bold', pad=15)

plt.tight_layout()
fig.savefig(os.path.join(TEMP_DIR, 'eda_wordcloud.png'), dpi=150, bbox_inches='tight')
print('Сохранено: eda_wordcloud.png')
plt.show()
plt.close(fig)

In [ ]:
# 9.6 Сводная статистика

# считаем общее количество файлов в директории texts/
texts_dir = os.path.join(BASE_DIR, 'texts')
all_files_count = len([f for f in os.listdir(texts_dir) if os.path.isfile(os.path.join(texts_dir, f))])

# количество уникальных авторов
unique_authors = meta['author'].nunique()

# количество текстов по жанрам
genre_dist_eda = meta['genre'].value_counts()

# общий word_count
total_words_eda = meta['word_count'].sum()

# собираем все в словарь
stats = {
    'Всего файлов в texts/': all_files_count,
    'Файлов после фильтрации': len(meta),
    'Общий объем (слова)': int(total_words_eda),
    'Среднее слов на текст': round(mean_wc, 1),
    'Медиана слов на текст': round(median_wc, 1),
    'Минимум слов': int(word_counts_eda.min()),
    'Максимум слов': int(word_counts_eda.max()),
    'Уникальных авторов': unique_authors,
    'Текстов с годом': len(with_year),
    'Текстов без года': int(without_year_count),
}

# добавляем количество по жанрам
for genre_name, count in genre_dist_eda.items():
    # переводим жанр на русский
    ru_name = genre_ru.get(genre_name, genre_name)
    stats[f'Жанр: {ru_name}'] = count

# печатаем сводку
print('Сводная статистика корпуса:')
print()
max_key_len = max(len(k) for k in stats.keys())
for key, val in stats.items():
    if isinstance(val, float):
        print(f'  {key:<{max_key_len}}  {val:>12,.1f}')
    else:
        print(f'  {key:<{max_key_len}}  {val:>12,}')
print()

# сохраняем сводку в CSV
stats_df = pd.DataFrame(list(stats.items()), columns=['metric', 'value'])
stats_csv_path = os.path.join(DATASETS_DIR, 'corpus_statistics.csv')
stats_df.to_csv(stats_csv_path, index=False, encoding='utf-8-sig')
print(f'Сводная статистика сохранена: {stats_csv_path}')

In [ ]:
# 9.7 Boxplot объема по жанрам

# группируем word_count по жанрам
genres_list = list(genre_ru.keys())
genre_data = [meta[meta['genre'] == g]['word_count'].values for g in genres_list]
genre_labels_ru = [genre_ru[g] for g in genres_list]

fig, ax = plt.subplots(figsize=(10, 6))

# рисуем boxplot
bp = ax.boxplot(
    genre_data,
    tick_labels=genre_labels_ru,
    patch_artist=True,
    medianprops=dict(color='black', linewidth=2)
)

# раскрашиваем боксы
for patch, color in zip(bp['boxes'], colors_eda):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Количество слов', fontsize=13)
ax.set_title('Распределение объема текстов по жанрам', fontsize=15, fontweight='bold')

plt.tight_layout()
fig.savefig(os.path.join(TEMP_DIR, 'eda_genre_boxplot.png'), dpi=150, bbox_inches='tight')
print('Сохранено: eda_genre_boxplot.png')
plt.show()
plt.close(fig)

In [ ]:
# список всех сохраненных визуализаций EDA
print('Все сохраненные визуализации:')
saved_files = [
    'eda_genre_pie.png',
    'eda_word_count_hist.png',
    'eda_top20_authors.png',
    'eda_year_scatter.png',
    'eda_wordcloud.png',
    'eda_genre_boxplot.png',
]
for f in saved_files:
    full_path = os.path.join(TEMP_DIR, f)
    exists = os.path.exists(full_path)
    size_kb = os.path.getsize(full_path) / 1024 if exists else 0
    status = f'{size_kb:.0f} KB' if exists else 'НЕ НАЙДЕН'
    print(f'  {f}: {status}')
print()
print(f'Сводная статистика: {stats_csv_path}')
print()
print('EDA завершен.')

## Выводы по подготовке данных

Подготовка корпуса завершена. Вот что мы сделали и получили:

- Из 128 файлов в директории `texts/` отобрали **98 оригинальных македонских художественных текстов** (отсеяли переводные, албанские, wikipedia, новостные и научные).

- Общий объём корпуса: **4.6 миллиона слов** -- этого достаточно для обучения модели.

- Жанровый состав: 45 поэзия, 40 проза, 9 драма, 4 детская литература.

- Разбиение на выборки:
  - **train**: 69 текстов (~3.32M слов)
  - **validation**: 14 текстов (~596K слов)
  - **test**: 15 текстов (~717K слов)

- Все датасеты сохранены в `datasets/` и готовы для следующего этапа -- лемматизация и построение графов.